# Build Topology Features and Full HeteroData

This notebook builds the first **full FREIGHT-MNet heterogeneous graph input** for the D-CQHGT line of experiments.

The previous Cold-OD experiments showed that FAF-zone graph-style representation is highly useful, but the FAF-zone-only edge schema is not enough to claim that physical multimodal topology is the main causal source of improvement. This notebook therefore moves from the FAF-zone-only graph to a richer multimodal graph with:

- `faf_zone` nodes.
- `terminal` nodes.
- Mode-specific FAF demand edges.
- Train-only supervised OD message edges.
- Coarse truck adjacency edges from NTAD FAF highway links.
- Coarse rail adjacency edges from NARN rail lines.
- FAF-zone-to-terminal access edges.
- OD-level topology features for the edge decoder.

The output is a PyTorch Geometric `HeteroData` object plus parquet/CSV/JSON artifacts that can be consumed by the next model notebook, `DCQHGT_Topology_ColdOD.ipynb`.

## 1. Environment setup and imports

This cell imports the core packages and checks optional geospatial / graph dependencies. The notebook intentionally does **not** call `DataFrame.to_markdown()` and does not require `tabulate`.

If `geopandas`, `networkx`, `torch`, or `torch_geometric` is missing, install the package in the active Jupyter kernel environment, not in a different base environment.

In [1]:
import json
import math
import os
import random
import warnings
from dataclasses import asdict, dataclass
from itertools import combinations
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

# Disable pandas optional acceleration modules that previously caused NumPy ABI conflicts.
pd.set_option("compute.use_numexpr", False)
pd.set_option("compute.use_bottleneck", False)

try:
    import geopandas as gpd
except Exception as exc:
    raise ImportError(
        "geopandas is required for terminal assignment and topology edge construction. "
        "Install it in the active Jupyter kernel environment."
    ) from exc

try:
    import networkx as nx
except Exception as exc:
    raise ImportError(
        "networkx is required for shortest path topology features. "
        "Install it in the active Jupyter kernel environment."
    ) from exc

try:
    import torch
    from torch_geometric.data import HeteroData
except Exception as exc:
    raise ImportError(
        "torch and torch_geometric are required to build and save the HeteroData object. "
        "Install them in the active Jupyter kernel environment."
    ) from exc

print("Python executable:", os.sys.executable)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("geopandas:", gpd.__version__)
print("networkx:", nx.__version__)
print("torch:", torch.__version__)
print("torch_geometric HeteroData imported successfully.")

Python executable: E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Scripts\python.exe
numpy: 2.4.5
pandas: 2.3.3
geopandas: 1.1.3
networkx: 3.6.1
torch: 2.12.0+cu126
torch_geometric HeteroData imported successfully.


## 2. Experiment configuration

Update only this configuration cell when changing scope, input roots, or topology-processing options.

The default paths match the current project layout:

```text
E:\NetworkOptimization\pythonProject1\Data
```

The topology processing can be expensive because FAF highway and NARN rail line chunks are large. For a smoke test, set `max_truck_topology_files` or `max_rail_topology_files` to a small integer such as `5`. For the final build, keep them as `None`.

In [2]:
@dataclass
class BuildConfig:
    """Configuration for topology feature construction and full HeteroData export."""

    data_root: Path = Path(r"E:\NetworkOptimization\pythonProject1\Data")
    scope: str = "east_plus_gulf"
    run_name: str = "build_topology_features_full_heterodata_v1"
    seed: int = 42

    # Topology processing options.
    projected_crs: str = "EPSG:5070"  # CONUS Albers; good for distance in meters.
    topology_predicate: str = "intersects"
    max_truck_topology_files: Optional[int] = None
    max_rail_topology_files: Optional[int] = None
    reuse_cached_topology_edges: bool = True
    overwrite_outputs: bool = True

    # Edge construction policy.
    message_edge_policy: str = "cold_train_if_available_else_temporal_train"
    include_reverse_demand_edges: bool = True
    include_reverse_train_od_edges: bool = True
    include_self_loops: bool = True

    # Distance feature handling.
    unreachable_distance_fill_strategy: str = "train_median"  # train_median or zero
    terminal_nearest_distance_fill_miles: float = 9999.0

    # Output options.
    save_heterodata_pt: bool = True
    save_debug_tables: bool = True


cfg = BuildConfig()
random.seed(cfg.seed)
np.random.seed(cfg.seed)

print(cfg)

BuildConfig(data_root=WindowsPath('E:/NetworkOptimization/pythonProject1/Data'), scope='east_plus_gulf', run_name='build_topology_features_full_heterodata_v1', seed=42, projected_crs='EPSG:5070', topology_predicate='intersects', max_truck_topology_files=None, max_rail_topology_files=None, reuse_cached_topology_edges=True, overwrite_outputs=True, message_edge_policy='cold_train_if_available_else_temporal_train', include_reverse_demand_edges=True, include_reverse_train_od_edges=True, include_self_loops=True, unreachable_distance_fill_strategy='train_median', terminal_nearest_distance_fill_miles=9999.0, save_heterodata_pt=True, save_debug_tables=True)


## 3. Path management

This cell resolves all required input paths and all output paths. It also creates output directories.

The notebook writes the final PyG object to `Data/08_processed/graph_inputs` so future model notebooks can reuse it without rebuilding topology.

In [3]:
@dataclass
class BuildPaths:
    """Resolved input and output paths for the topology/HeteroData build."""

    data_root: Path
    supervised_path: Path
    manifest_path: Path
    faf_regions_path: Path
    east_plus_gulf_faf_zones_path: Path
    terminals_path: Path
    truck_links_dir: Path
    rail_lines_dir: Path
    cold_baseline_all_splits_path: Path
    output_dir: Path
    graph_inputs_dir: Path
    tables_dir: Path
    cache_dir: Path
    heterodata_path: Path
    topology_features_path: Path
    metadata_path: Path


def get_scope_suffix(scope: str) -> str:
    """Convert a user-facing scope value into the filename suffix used by the data pipeline."""
    normalized = scope.strip().lower().replace("-", "_")
    if normalized in {"east_plus_gulf", "eastplusgulf"}:
        return "east_plus_gulf"
    if normalized in {"all", "all_valid_faf_zones"}:
        return "all"
    raise ValueError(f"Unsupported scope: {scope!r}")


def make_paths(config: BuildConfig) -> BuildPaths:
    """Create all input/output paths from the project data root."""
    data_root = Path(config.data_root)
    scope_suffix = get_scope_suffix(config.scope)

    supervised_path = (
        data_root
        / "08_processed"
        / "model_ready"
        / f"freight_mnet_supervised_edges_2018_2024_{scope_suffix}.parquet"
    )
    manifest_path = (
        data_root
        / "08_processed"
        / "model_ready"
        / "_metadata"
        / "freight_mnet_supervised_feature_manifest.json"
    )

    faf_regions_path = data_root / "03_ntad_geospatial" / "faf_regions" / "faf_regions_v2.parquet"
    east_plus_gulf_faf_zones_path = (
        data_root / "09_crosswalks" / "county_to_faf" / "east_plus_gulf_faf_zones.parquet"
    )
    terminals_path = (
        data_root
        / "03_ntad_geospatial"
        / "intermodal_rail_tofc_cofc"
        / "intermodal_rail_tofc_cofc_v2.parquet"
    )
    truck_links_dir = data_root / "03_ntad_geospatial" / "faf_network_links" / "parquet_chunks_v2"
    rail_lines_dir = data_root / "03_ntad_geospatial" / "narn_lines" / "parquet_chunks_v2"

    cold_baseline_all_splits_path = (
        data_root
        / "10_experiments"
        / "cold_od_split_baselines_v1_notebook"
        / scope_suffix
        / "predictions_cold_od_all_splits.parquet"
    )

    output_dir = data_root / "10_experiments" / config.run_name / scope_suffix
    graph_inputs_dir = data_root / "08_processed" / "graph_inputs"
    tables_dir = output_dir / "tables"
    cache_dir = output_dir / "cache"

    heterodata_path = graph_inputs_dir / f"freight_mnet_full_heterodata_{scope_suffix}.pt"
    topology_features_path = graph_inputs_dir / f"topology_features_od_{scope_suffix}.parquet"
    metadata_path = graph_inputs_dir / f"freight_mnet_full_heterodata_{scope_suffix}.metadata.json"

    for directory in [output_dir, graph_inputs_dir, tables_dir, cache_dir]:
        directory.mkdir(parents=True, exist_ok=True)

    return BuildPaths(
        data_root=data_root,
        supervised_path=supervised_path,
        manifest_path=manifest_path,
        faf_regions_path=faf_regions_path,
        east_plus_gulf_faf_zones_path=east_plus_gulf_faf_zones_path,
        terminals_path=terminals_path,
        truck_links_dir=truck_links_dir,
        rail_lines_dir=rail_lines_dir,
        cold_baseline_all_splits_path=cold_baseline_all_splits_path,
        output_dir=output_dir,
        graph_inputs_dir=graph_inputs_dir,
        tables_dir=tables_dir,
        cache_dir=cache_dir,
        heterodata_path=heterodata_path,
        topology_features_path=topology_features_path,
        metadata_path=metadata_path,
    )


paths = make_paths(cfg)
print(json.dumps({key: str(value) for key, value in asdict(paths).items()}, indent=2))

{
  "data_root": "E:\\NetworkOptimization\\pythonProject1\\Data",
  "supervised_path": "E:\\NetworkOptimization\\pythonProject1\\Data\\08_processed\\model_ready\\freight_mnet_supervised_edges_2018_2024_east_plus_gulf.parquet",
  "manifest_path": "E:\\NetworkOptimization\\pythonProject1\\Data\\08_processed\\model_ready\\_metadata\\freight_mnet_supervised_feature_manifest.json",
  "faf_regions_path": "E:\\NetworkOptimization\\pythonProject1\\Data\\03_ntad_geospatial\\faf_regions\\faf_regions_v2.parquet",
  "east_plus_gulf_faf_zones_path": "E:\\NetworkOptimization\\pythonProject1\\Data\\09_crosswalks\\county_to_faf\\east_plus_gulf_faf_zones.parquet",
  "terminals_path": "E:\\NetworkOptimization\\pythonProject1\\Data\\03_ntad_geospatial\\intermodal_rail_tofc_cofc\\intermodal_rail_tofc_cofc_v2.parquet",
  "truck_links_dir": "E:\\NetworkOptimization\\pythonProject1\\Data\\03_ntad_geospatial\\faf_network_links\\parquet_chunks_v2",
  "rail_lines_dir": "E:\\NetworkOptimization\\pythonProject1\\

## 4. General utility functions

These helpers handle FAF code normalization, schema discovery, Parquet-safe writing, and basic validation.

FAF zone IDs are normalized to three-character strings such as `011`, `012`, and `089` to avoid merge errors between integer and string zone codes.

In [4]:
LABEL_COLUMNS = ["truck_q25", "truck_q50", "truck_q75"]
TAUS = np.array([0.25, 0.50, 0.75], dtype=np.float32)

ORIGIN_CANDIDATES = ["faf_orig", "orig_faf", "origin_faf", "origin", "orig", "o_faf", "from_faf"]
DEST_CANDIDATES = ["faf_dest", "dest_faf", "destination_faf", "destination", "dest", "d_faf", "to_faf"]
YEAR_CANDIDATES = ["year", "Year", "yr"]
FAF_ZONE_CANDIDATES = ["faf_zone", "FAF_ZONE", "faf", "FAF", "faf_id", "FAF_ID", "zone", "ZONE", "id"]


def normalize_faf_code(value: object) -> Optional[str]:
    """Normalize a FAF zone code to a three-character string."""
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except TypeError:
        pass

    text = str(value).strip()
    if not text:
        return None

    # Handle values like "11.0" safely.
    try:
        number = int(float(text))
        return f"{number:03d}"
    except ValueError:
        return text.zfill(3) if text.isdigit() else text


def find_first_existing_column(columns: Iterable[str], candidates: Sequence[str], required: bool = True) -> Optional[str]:
    """Return the first candidate column found in a dataframe-like column list."""
    column_set = set(columns)
    for candidate in candidates:
        if candidate in column_set:
            return candidate
    if required:
        raise ValueError(f"None of the candidate columns were found: {candidates}")
    return None


def ensure_file_exists(path: Path, label: str) -> None:
    """Raise a helpful error if a required file is missing."""
    if not Path(path).exists():
        raise FileNotFoundError(f"Missing {label}: {path}")


def list_parquet_files(path: Path, max_files: Optional[int] = None) -> List[Path]:
    """Return sorted parquet files from a file path or directory."""
    path = Path(path)
    if path.is_file():
        files = [path]
    elif path.is_dir():
        files = sorted(path.glob("*.parquet"))
    else:
        files = []

    if max_files is not None:
        files = files[: int(max_files)]
    return files


def make_unique_columns(columns: Iterable[object]) -> List[str]:
    """Return unique string column names while preserving order."""
    seen: Dict[str, int] = {}
    result: List[str] = []
    for column in columns:
        base = str(column)
        count = seen.get(base, 0)
        result.append(base if count == 0 else f"{base}__dup{count}")
        seen[base] = count + 1
    return result


def normalize_dataframe_for_parquet(frame: pd.DataFrame) -> pd.DataFrame:
    """Normalize object columns so pyarrow can write Parquet reliably."""
    clean = frame.copy()
    clean.columns = make_unique_columns(clean.columns)
    for column in clean.columns:
        series = clean[column]
        if pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series):
            clean[column] = series.map(lambda x: pd.NA if x is None or (not isinstance(x, (list, dict, tuple, set)) and pd.isna(x)) else str(x)).astype("string")
    return clean


def safe_to_parquet(frame: pd.DataFrame, path: Path) -> Path:
    """Write a dataframe to Parquet after object-column normalization."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    normalize_dataframe_for_parquet(frame).to_parquet(path, index=False, engine="pyarrow")
    return path


def write_json(payload: dict, path: Path) -> Path:
    """Write a JSON artifact with robust serialization of NumPy/Pandas objects."""
    def default(value):
        if isinstance(value, (np.integer,)):
            return int(value)
        if isinstance(value, (np.floating,)):
            return float(value)
        if isinstance(value, np.ndarray):
            return value.tolist()
        if isinstance(value, Path):
            return str(value)
        if isinstance(value, pd.Timestamp):
            return value.isoformat()
        return str(value)

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        json.dump(payload, file, indent=2, default=default)
    return path


def log_frame(name: str, frame: pd.DataFrame, max_rows: int = 5) -> None:
    """Print a compact dataframe preview without requiring tabulate."""
    print(f"\n{name}: shape={frame.shape}")
    if not frame.empty:
        print(frame.head(max_rows).to_string(index=False))

## 5. Load supervised table and feature manifest

The supervised table provides the OD-year labels and the 64 current OD/zone features. The feature manifest is used as the source of truth for predictive columns so that diagnostic columns such as `obs_weight_sum` and `n_fmi_county_pairs` are not accidentally fed into the model.

In [5]:
ensure_file_exists(paths.supervised_path, "model-ready supervised edge table")
ensure_file_exists(paths.manifest_path, "feature manifest")

with paths.manifest_path.open("r", encoding="utf-8") as file:
    manifest = json.load(file)

manifest_feature_columns = manifest.get("feature_columns") or manifest.get("features")
if not manifest_feature_columns:
    raise ValueError("The feature manifest does not contain 'feature_columns'.")
manifest_feature_columns = list(manifest_feature_columns)

supervised_df = pd.read_parquet(paths.supervised_path)
supervised_df.columns = make_unique_columns(supervised_df.columns)

origin_col = find_first_existing_column(supervised_df.columns, ORIGIN_CANDIDATES)
dest_col = find_first_existing_column(supervised_df.columns, DEST_CANDIDATES)
year_col = find_first_existing_column(supervised_df.columns, YEAR_CANDIDATES)

missing_labels = [column for column in LABEL_COLUMNS if column not in supervised_df.columns]
missing_features = [column for column in manifest_feature_columns if column not in supervised_df.columns]
if missing_labels:
    raise ValueError(f"Missing label columns: {missing_labels}")
if missing_features:
    raise ValueError(f"Manifest features missing from supervised table: {missing_features[:20]}")

supervised_df["faf_orig_str"] = supervised_df[origin_col].map(normalize_faf_code)
supervised_df["faf_dest_str"] = supervised_df[dest_col].map(normalize_faf_code)
supervised_df["year_int"] = supervised_df[year_col].astype(int)

for column in LABEL_COLUMNS + manifest_feature_columns:
    supervised_df[column] = pd.to_numeric(supervised_df[column], errors="coerce")

# Basic label monotonicity guard.
violations = ((supervised_df["truck_q25"] > supervised_df["truck_q50"]) | (supervised_df["truck_q50"] > supervised_df["truck_q75"])).sum()
if violations:
    raise ValueError(f"Found {violations} rows with non-monotone labels.")

supervised_df["temporal_split"] = np.select(
    [
        supervised_df["year_int"].between(2018, 2022),
        supervised_df["year_int"].eq(2023),
        supervised_df["year_int"].eq(2024),
    ],
    ["train", "val", "test"],
    default="unused",
)

print("Supervised table shape:", supervised_df.shape)
print("Origin column:", origin_col, "Destination column:", dest_col, "Year column:", year_col)
print("Manifest feature count:", len(manifest_feature_columns))
print("Temporal split counts:")
print(supervised_df["temporal_split"].value_counts().to_string())

Supervised table shape: (73972, 96)
Origin column: faf_orig Destination column: faf_dest Year column: year
Manifest feature count: 64
Temporal split counts:
temporal_split
train    52773
val      10625
test     10574


## 6. Load or reconstruct Cold-OD split assignments

For topology and graph modeling, we preserve both temporal splits and Cold-OD splits. If the previous Cold-OD baseline prediction artifact exists, this cell imports its split assignments. Otherwise, it builds a deterministic fallback Cold-OD split from OD pairs.

The fallback split is designed for reproducibility, but the preferred path is to reuse the already established Cold-OD split from the previous baseline notebook.

In [6]:
def build_deterministic_cold_od_split(frame: pd.DataFrame, val_fraction: float = 0.10, test_fraction: float = 0.10, seed: int = 42) -> pd.Series:
    """Build a deterministic OD-pair-level Cold-OD split if prior split artifacts are unavailable."""
    unique_pairs = frame[["faf_orig_str", "faf_dest_str"]].drop_duplicates().reset_index(drop=True)
    rng = np.random.default_rng(seed)
    permuted_indices = rng.permutation(len(unique_pairs))

    n_val = int(round(len(unique_pairs) * val_fraction))
    n_test = int(round(len(unique_pairs) * test_fraction))
    val_pairs = set(map(tuple, unique_pairs.iloc[permuted_indices[:n_val]].to_numpy()))
    test_pairs = set(map(tuple, unique_pairs.iloc[permuted_indices[n_val : n_val + n_test]].to_numpy()))

    cold_split = []
    for row in frame[["faf_orig_str", "faf_dest_str", "year_int"]].itertuples(index=False):
        pair = (row.faf_orig_str, row.faf_dest_str)
        year = int(row.year_int)
        if pair in val_pairs and year == 2023:
            cold_split.append("val")
        elif pair in test_pairs and year == 2024:
            cold_split.append("test")
        elif pair not in val_pairs and pair not in test_pairs and 2018 <= year <= 2022:
            cold_split.append("train")
        else:
            cold_split.append("unused")
    return pd.Series(cold_split, index=frame.index, name="cold_split")


def load_cold_split_from_predictions(frame: pd.DataFrame, prediction_path: Path) -> Optional[pd.Series]:
    """Load Cold-OD split assignments from a prior prediction parquet if available."""
    if not Path(prediction_path).exists():
        return None

    predictions = pd.read_parquet(prediction_path)
    required = {"split", "year"}
    if not required.issubset(predictions.columns):
        return None

    pred_origin_col = find_first_existing_column(predictions.columns, ORIGIN_CANDIDATES + ["faf_orig_str"], required=False)
    pred_dest_col = find_first_existing_column(predictions.columns, DEST_CANDIDATES + ["faf_dest_str"], required=False)
    if pred_origin_col is None or pred_dest_col is None:
        return None

    split_table = predictions[[pred_origin_col, pred_dest_col, "year", "split"]].drop_duplicates().copy()
    split_table["faf_orig_str"] = split_table[pred_origin_col].map(normalize_faf_code)
    split_table["faf_dest_str"] = split_table[pred_dest_col].map(normalize_faf_code)
    split_table["year_int"] = split_table["year"].astype(int)
    split_table = split_table[["faf_orig_str", "faf_dest_str", "year_int", "split"]].drop_duplicates()

    conflict_count = split_table.groupby(["faf_orig_str", "faf_dest_str", "year_int"])["split"].nunique().gt(1).sum()
    if conflict_count:
        raise ValueError(f"Cold split artifact has {conflict_count} conflicting split assignments.")

    split_table = split_table.drop_duplicates(["faf_orig_str", "faf_dest_str", "year_int"])
    merged = frame[["faf_orig_str", "faf_dest_str", "year_int"]].merge(
        split_table,
        on=["faf_orig_str", "faf_dest_str", "year_int"],
        how="left",
    )
    return merged["split"].fillna("unused").astype(str).rename("cold_split")


cold_split_series = load_cold_split_from_predictions(supervised_df, paths.cold_baseline_all_splits_path)
if cold_split_series is None:
    warnings.warn("Cold-OD split artifact not found. Building deterministic fallback split.")
    cold_split_series = build_deterministic_cold_od_split(supervised_df, seed=cfg.seed)

supervised_df["cold_split"] = cold_split_series.values

# Verify OD-pair disjointness for Cold-OD split.
def od_pair_set(frame: pd.DataFrame, split_name: str) -> set:
    subset = frame.loc[frame["cold_split"].eq(split_name), ["faf_orig_str", "faf_dest_str"]].drop_duplicates()
    return set(map(tuple, subset.to_numpy()))

train_pairs = od_pair_set(supervised_df, "train")
val_pairs = od_pair_set(supervised_df, "val")
test_pairs = od_pair_set(supervised_df, "test")

assert train_pairs.isdisjoint(val_pairs), "Cold-OD leakage: train and val OD pairs overlap."
assert train_pairs.isdisjoint(test_pairs), "Cold-OD leakage: train and test OD pairs overlap."
assert val_pairs.isdisjoint(test_pairs), "Cold-OD leakage: val and test OD pairs overlap."

print("Cold split counts:")
print(supervised_df["cold_split"].value_counts().to_string())
print("Cold OD pairs:", {"train": len(train_pairs), "val": len(val_pairs), "test": len(test_pairs)})

Cold split counts:
cold_split
train     42849
unused    29109
test       1057
val         957
Cold OD pairs: {'train': 8748, 'val': 957, 'test': 1057}


## 7. Load FAF regions and define the FAF node universe

FAF regions provide the polygon geometry used to connect terminals and physical truck/rail topology to FAF zones. The graph includes all zones that appear in the supervised table and all zones available in the FAF region file.

In [10]:
# ---------------------------------------------------------------------
# Load FAF regions and infer the FAF zone identifier robustly
# ---------------------------------------------------------------------
#
# This cell replaces the original FAF-region loading cell.
#
# The previous version failed because the FAF regions GeoParquet did not
# contain any column exactly matching the predefined FAF_ZONE_CANDIDATES.
# This version performs robust column inference using:
#   1. manual override if provided,
#   2. case-insensitive exact matching,
#   3. normalized column-name matching,
#   4. value overlap with FAF zones observed in the supervised table,
#   5. index fallback if the zone code was stored as the GeoDataFrame index.
#
# It still keeps the downstream contract:
#   faf_regions: GeoDataFrame with columns ["faf_zone_str", geometry]
#   supervised_zones: set[str]
#   region_zones: set[str]
#   all_faf_zones: list[str]
#   zone_to_idx / idx_to_zone: graph node maps

import re
from typing import Optional, Sequence

ensure_file_exists(paths.faf_regions_path, "FAF regions GeoParquet")

faf_regions_raw = gpd.read_parquet(paths.faf_regions_path)

if faf_regions_raw.empty:
    raise ValueError("FAF regions file is empty.")

if not isinstance(faf_regions_raw, gpd.GeoDataFrame):
    raise TypeError(
        "FAF regions file was loaded, but it is not a GeoDataFrame. "
        "Please check that the GeoParquet file contains a valid geometry column."
    )

geometry_col = faf_regions_raw.geometry.name
if geometry_col not in faf_regions_raw.columns:
    raise ValueError(
        "FAF regions file does not contain an active geometry column. "
        f"Detected geometry name: {geometry_col}. "
        f"Available columns: {list(faf_regions_raw.columns)}"
    )

# Build the supervised-zone universe first. This gives us a strong signal
# for choosing the correct FAF-zone column in the region file.
supervised_zones = (
    set(supervised_df["faf_orig_str"].dropna().astype(str))
    | set(supervised_df["faf_dest_str"].dropna().astype(str))
)

# Optional manual override. If automatic inference still fails, set this to
# the actual FAF zone column name after inspecting the printed column list.
manual_faf_zone_col: Optional[str] = None


def normalize_text_for_matching(value: object) -> str:
    """Normalize a column name for loose matching."""
    return re.sub(r"[^a-z0-9]+", "", str(value).strip().lower())


def normalize_faf_code_robust(value: object) -> Optional[str]:
    """
    Convert a FAF-zone-like value into a three-character string.

    Examples:
        1          -> "001"
        "1"        -> "001"
        "011"      -> "011"
        "FAF 11"   -> "011"
        "Zone_089" -> "089"

    Returns None when no plausible 1-3 digit FAF code can be extracted.
    """
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    # First try the project's existing normalizer.
    try:
        code = normalize_faf_code(value)
        if code is not None:
            code_text = str(code).strip()
            if code_text and code_text.lower() not in {"nan", "none"}:
                digits = re.findall(r"\d+", code_text)
                if digits:
                    return digits[-1].zfill(3)
    except Exception:
        pass

    text = str(value).strip()
    if not text:
        return None

    # Extract the most likely short numeric token.
    digit_tokens = re.findall(r"(?<!\d)(\d{1,3})(?!\d)", text)
    if not digit_tokens:
        return None

    return digit_tokens[-1].zfill(3)


def score_faf_zone_column(
    frame: pd.DataFrame,
    column: str,
    supervised_zone_set: set[str],
) -> dict:
    """
    Score a possible FAF-zone column.

    A good column should:
      - have a name related to FAF / zone / region,
      - convert many values into 3-digit codes,
      - overlap with supervised FAF zones.
    """
    normalized_name = normalize_text_for_matching(column)

    name_bonus = 0
    if normalized_name in {
        "fafzone",
        "fafzoneid",
        "fafregion",
        "fafregionid",
        "fafid",
        "faf",
        "zone",
        "zoneid",
        "region",
        "regionid",
    }:
        name_bonus += 100

    if "faf" in normalized_name:
        name_bonus += 50
    if "zone" in normalized_name:
        name_bonus += 30
    if "region" in normalized_name:
        name_bonus += 20
    if "id" in normalized_name:
        name_bonus += 5

    codes = frame[column].map(normalize_faf_code_robust)
    non_null_codes = codes.dropna()
    unique_codes = set(non_null_codes.astype(str))
    overlap = unique_codes & supervised_zone_set

    valid_rate = float(len(non_null_codes) / max(len(frame), 1))
    overlap_count = len(overlap)
    unique_count = len(unique_codes)

    # High overlap is the strongest signal, but name match is also important
    # to avoid accidentally selecting OBJECTID-like columns.
    score = (
        overlap_count * 1000
        + name_bonus * 10
        + valid_rate * 100
        + min(unique_count, 200)
    )

    return {
        "column": column,
        "normalized_name": normalized_name,
        "score": score,
        "name_bonus": name_bonus,
        "valid_rate": valid_rate,
        "unique_code_count": unique_count,
        "overlap_with_supervised_zones": overlap_count,
        "example_values": list(frame[column].dropna().astype(str).head(5)),
    }


def infer_faf_zone_column(
    frame: pd.DataFrame,
    supervised_zone_set: set[str],
    manual_column: Optional[str] = None,
) -> tuple[str, pd.DataFrame]:
    """
    Infer the FAF-zone identifier column.

    Returns:
        selected_column:
            A dataframe column name, or "__index__" if the index is used.
        audit_table:
            Candidate scoring table for debugging.
    """
    if manual_column is not None:
        if manual_column not in frame.columns:
            raise ValueError(
                f"manual_faf_zone_col='{manual_column}' was not found. "
                f"Available columns: {list(frame.columns)}"
            )
        audit = pd.DataFrame(
            [score_faf_zone_column(frame, manual_column, supervised_zone_set)]
        )
        return manual_column, audit

    # Expanded candidates: includes common naming variants found in NTAD/FAF files.
    expanded_candidates = list(dict.fromkeys(
        list(FAF_ZONE_CANDIDATES)
        + [
            "FAF_Zone",
            "FAF Zone",
            "FAFZone",
            "FAFZONE",
            "FAF_Zone_ID",
            "FAF_ZONE_ID",
            "faf_zone_id",
            "FAFREGION",
            "FAF_REGION",
            "FAF_Region",
            "faf_region",
            "FAFRegion",
            "FAFREGION_ID",
            "fafregion_id",
            "FAFID",
            "FAF_ID",
            "FAF_CODE",
            "faf_code",
            "ZoneID",
            "zone_id",
            "ZONE_ID",
            "RegionID",
            "region_id",
            "REGION_ID",
            "Region",
            "REGION",
            "Name",
            "NAME",
            "namelsad",
            "NAMELSAD",
        ]
    ))

    # First pass: exact match against expanded candidates.
    for candidate in expanded_candidates:
        if candidate in frame.columns:
            audit = pd.DataFrame(
                [score_faf_zone_column(frame, candidate, supervised_zone_set)]
            )
            return candidate, audit

    # Second pass: normalized exact matching.
    normalized_to_column = {
        normalize_text_for_matching(column): column
        for column in frame.columns
        if column != geometry_col
    }
    for candidate in expanded_candidates:
        normalized_candidate = normalize_text_for_matching(candidate)
        if normalized_candidate in normalized_to_column:
            selected = normalized_to_column[normalized_candidate]
            audit = pd.DataFrame(
                [score_faf_zone_column(frame, selected, supervised_zone_set)]
            )
            return selected, audit

    # Third pass: score all non-geometry columns.
    scored_rows = []
    for column in frame.columns:
        if column == geometry_col:
            continue
        scored_rows.append(score_faf_zone_column(frame, column, supervised_zone_set))

    audit_table = pd.DataFrame(scored_rows).sort_values(
        ["score", "overlap_with_supervised_zones", "name_bonus"],
        ascending=False,
    )

    if not audit_table.empty:
        best = audit_table.iloc[0]
        # Accept automatically if the best column has either a strong name signal
        # or non-trivial overlap with the supervised FAF zones.
        if (
            best["overlap_with_supervised_zones"] > 0
            or best["name_bonus"] >= 50
        ):
            return str(best["column"]), audit_table

    # Fourth pass: index fallback. Some GeoParquet files preserve FAF code as index.
    index_codes = pd.Series(frame.index, index=frame.index).map(normalize_faf_code_robust)
    index_unique_codes = set(index_codes.dropna().astype(str))
    index_overlap = index_unique_codes & supervised_zone_set

    if len(index_overlap) > 0:
        index_audit = pd.DataFrame(
            [
                {
                    "column": "__index__",
                    "normalized_name": str(frame.index.name),
                    "score": len(index_overlap) * 1000 + len(index_unique_codes),
                    "name_bonus": 0,
                    "valid_rate": float(index_codes.notna().mean()),
                    "unique_code_count": len(index_unique_codes),
                    "overlap_with_supervised_zones": len(index_overlap),
                    "example_values": list(pd.Series(frame.index).astype(str).head(5)),
                }
            ]
        )
        if audit_table.empty:
            audit_table = index_audit
        else:
            audit_table = pd.concat([audit_table, index_audit], ignore_index=True)
        return "__index__", audit_table.sort_values("score", ascending=False)

    # If everything fails, print a rich diagnostic and raise a clear error.
    print("Available FAF region columns:")
    for column in frame.columns:
        print(f"  - {column}")

    if not audit_table.empty:
        print("\nFAF-zone candidate score audit:")
        print(audit_table.head(20).to_string(index=False))

    raise ValueError(
        "Could not infer the FAF zone identifier column from faf_regions_v2.parquet. "
        "Set manual_faf_zone_col in this cell to the correct column name after "
        "inspecting the printed columns and candidate score audit."
    )


# Infer and apply FAF-zone column.
faf_zone_col, faf_zone_col_audit = infer_faf_zone_column(
    faf_regions_raw,
    supervised_zones,
    manual_column=manual_faf_zone_col,
)

print("Selected FAF zone column:", faf_zone_col)
print("\nFAF-zone column inference audit:")
print(faf_zone_col_audit.head(12).to_string(index=False))

faf_regions = faf_regions_raw.copy()

if faf_zone_col == "__index__":
    faf_regions["faf_zone_str"] = pd.Series(
        faf_regions.index,
        index=faf_regions.index,
    ).map(normalize_faf_code_robust)
else:
    faf_regions["faf_zone_str"] = faf_regions[faf_zone_col].map(normalize_faf_code_robust)

# Drop invalid zone IDs and invalid geometries.
faf_regions = faf_regions.dropna(subset=["faf_zone_str"]).copy()
faf_regions = faf_regions[faf_regions.geometry.notna()].copy()
faf_regions = faf_regions[~faf_regions.geometry.is_empty].copy()

if faf_regions.empty:
    raise ValueError(
        "After FAF-zone normalization and geometry filtering, no FAF regions remain. "
        f"Selected FAF zone column was: {faf_zone_col}"
    )

# Keep one polygon or multipolygon geometry per FAF zone if the file has duplicates.
faf_regions = (
    faf_regions[["faf_zone_str", geometry_col]]
    .dissolve(by="faf_zone_str", as_index=False)
)

region_zones = set(faf_regions["faf_zone_str"].dropna().astype(str))
all_faf_zones = sorted(supervised_zones | region_zones)
zone_to_idx = {zone: idx for idx, zone in enumerate(all_faf_zones)}
idx_to_zone = {idx: zone for zone, idx in zone_to_idx.items()}

missing_supervised_region_zones = sorted(supervised_zones - region_zones)
extra_region_zones = sorted(region_zones - supervised_zones)

print("\nFAF regions loaded successfully.")
print("FAF regions shape:", faf_regions.shape)
print("FAF zones in supervised table:", len(supervised_zones))
print("FAF zones in region file:", len(region_zones))
print("FAF nodes in full graph:", len(all_faf_zones))
print("Supervised zones missing polygon geometry:", len(missing_supervised_region_zones))
print("Region-file zones not used in supervised table:", len(extra_region_zones))

if missing_supervised_region_zones:
    print(
        "Warning: Some supervised FAF zones do not have polygon geometry in the "
        "region file. They will still be graph nodes, but terminal/topology "
        "features may be missing or imputed for those zones."
    )
    print("First missing supervised zones:", missing_supervised_region_zones[:20])

print("\nFAF regions preview:")
print(faf_regions.head().to_string(index=False))

Selected FAF zone column: FAF_Zone

FAF-zone column inference audit:
  column normalized_name    score  name_bonus  valid_rate  unique_code_count  overlap_with_supervised_zones            example_values
FAF_Zone         fafzone 106032.0         180         1.0                132                            104 [361, 131, 481, 241, 221]

FAF regions loaded successfully.
FAF regions shape: (132, 2)
FAF zones in supervised table: 104
FAF zones in region file: 132
FAF nodes in full graph: 132
Supervised zones missing polygon geometry: 0
Region-file zones not used in supervised table: 28

FAF regions preview:
faf_zone_str                                                                                                                                                                                                                                                                                                                                                                                         

## 8. Geospatial helpers for terminals and line-based topology

The functions below assign point/line features to FAF zones using spatial joins. Truck and rail adjacency are built by checking which FAF zones each line geometry intersects. If a line intersects multiple FAF zones, the notebook creates coarse zone-to-zone adjacency edges between those zones.

This is a coarse FAF-zone topology, not a link-level routing ground truth.

In [11]:
def ensure_geodataframe(frame: pd.DataFrame, lon_candidates: Sequence[str] = (), lat_candidates: Sequence[str] = (), crs: str = "EPSG:4326") -> gpd.GeoDataFrame:
    """Return a GeoDataFrame, creating point geometry from lon/lat if needed."""
    if isinstance(frame, gpd.GeoDataFrame) and frame.geometry.name in frame.columns:
        return frame

    lon_col = find_first_existing_column(frame.columns, lon_candidates, required=False)
    lat_col = find_first_existing_column(frame.columns, lat_candidates, required=False)
    if lon_col is None or lat_col is None:
        raise ValueError("Input does not contain geometry or usable longitude/latitude columns.")

    return gpd.GeoDataFrame(
        frame.copy(),
        geometry=gpd.points_from_xy(frame[lon_col].astype(float), frame[lat_col].astype(float)),
        crs=crs,
    )


def project_gdf(gdf: gpd.GeoDataFrame, target_crs: str) -> gpd.GeoDataFrame:
    """Project a GeoDataFrame to a target CRS, assigning WGS84 if CRS is missing."""
    clean = gdf.copy()
    if clean.crs is None:
        warnings.warn("GeoDataFrame has missing CRS. Assuming EPSG:4326.")
        clean = clean.set_crs("EPSG:4326")
    if str(clean.crs) != target_crs:
        clean = clean.to_crs(target_crs)
    return clean


def aggregate_undirected_edges(edge_rows: List[Tuple[str, str, float, str]]) -> pd.DataFrame:
    """Aggregate undirected zone-zone edges from raw pair rows."""
    if not edge_rows:
        return pd.DataFrame(columns=["src_faf", "dst_faf", "distance_miles", "edge_count", "source_relation"])

    raw = pd.DataFrame(edge_rows, columns=["src_faf", "dst_faf", "distance_miles", "source_relation"])
    raw = raw.dropna(subset=["src_faf", "dst_faf"])
    raw["src_faf"] = raw["src_faf"].map(normalize_faf_code)
    raw["dst_faf"] = raw["dst_faf"].map(normalize_faf_code)
    raw = raw[raw["src_faf"].ne(raw["dst_faf"])].copy()

    # Canonicalize undirected pairs before aggregation.
    canonical_src = np.minimum(raw["src_faf"].to_numpy(dtype=str), raw["dst_faf"].to_numpy(dtype=str))
    canonical_dst = np.maximum(raw["src_faf"].to_numpy(dtype=str), raw["dst_faf"].to_numpy(dtype=str))
    raw["u"] = canonical_src
    raw["v"] = canonical_dst

    aggregated = (
        raw.groupby(["u", "v", "source_relation"], as_index=False)
        .agg(distance_miles=("distance_miles", "median"), min_distance_miles=("distance_miles", "min"), edge_count=("distance_miles", "size"))
        .rename(columns={"u": "src_faf", "v": "dst_faf"})
    )
    return aggregated


def build_zone_adjacency_from_line_chunks(
    chunk_dir: Path,
    faf_regions_gdf: gpd.GeoDataFrame,
    relation_name: str,
    cache_path: Path,
    max_files: Optional[int] = None,
    reuse_cache: bool = True,
    predicate: str = "intersects",
    projected_crs: str = "EPSG:5070",
) -> pd.DataFrame:
    """Build coarse FAF-zone adjacency edges from geospatial line chunks."""
    if reuse_cache and Path(cache_path).exists():
        print(f"Loading cached {relation_name} adjacency:", cache_path)
        return pd.read_parquet(cache_path)

    files = list_parquet_files(chunk_dir, max_files=max_files)
    if not files:
        warnings.warn(f"No parquet files found for {relation_name}: {chunk_dir}")
        return pd.DataFrame(columns=["src_faf", "dst_faf", "distance_miles", "min_distance_miles", "edge_count", "source_relation"])

    regions_projected = project_gdf(faf_regions_gdf[["faf_zone_str", "geometry"]], projected_crs)
    edge_rows: List[Tuple[str, str, float, str]] = []

    for file_idx, file_path in enumerate(files, start=1):
        print(f"[{relation_name}] processing {file_idx}/{len(files)}: {file_path.name}")
        try:
            lines = gpd.read_parquet(file_path)
        except Exception as exc:
            warnings.warn(f"Skipping {file_path} because geopandas could not read it: {exc}")
            continue

        if lines.empty or lines.geometry.name not in lines.columns:
            continue

        lines = lines.dropna(subset=[lines.geometry.name]).copy()
        if lines.empty:
            continue

        lines = project_gdf(lines, projected_crs)
        lines = lines.reset_index(drop=True)
        lines["line_id_local"] = np.arange(len(lines), dtype=np.int64)
        lines["distance_miles"] = lines.geometry.length.astype(float) / 1609.344

        try:
            joined = gpd.sjoin(
                lines[["line_id_local", "distance_miles", "geometry"]],
                regions_projected[["faf_zone_str", "geometry"]],
                how="inner",
                predicate=predicate,
            )
        except Exception as exc:
            warnings.warn(f"Spatial join failed for {file_path}: {exc}")
            continue

        if joined.empty:
            continue

        for line_id, group in joined.groupby("line_id_local"):
            zones = sorted(set(group["faf_zone_str"].dropna().map(normalize_faf_code)))
            if len(zones) < 2:
                continue
            distance = float(group["distance_miles"].iloc[0])
            for src, dst in combinations(zones, 2):
                edge_rows.append((src, dst, distance, relation_name))

    adjacency = aggregate_undirected_edges(edge_rows)
    if not adjacency.empty:
        safe_to_parquet(adjacency, cache_path)
    return adjacency

## 9. Build terminal access edges

Intermodal terminals are assigned to FAF regions by point-in-polygon spatial join. If a terminal is not inside any FAF region, the nearest FAF region is used as a fallback. The output includes both access edges and terminal node features.

In [12]:
TERMINAL_ID_CANDIDATES = ["terminal_id", "TERM_ID", "OBJECTID", "objectid", "id", "ID", "name", "NAME", "terminal_name"]
LON_CANDIDATES = ["lon", "longitude", "LONGITUDE", "x", "X"]
LAT_CANDIDATES = ["lat", "latitude", "LATITUDE", "y", "Y"]


def build_terminal_assignments(terminals_path: Path, faf_regions_gdf: gpd.GeoDataFrame, projected_crs: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Assign terminals to FAF zones and return terminal features plus access edge table."""
    if not Path(terminals_path).exists():
        warnings.warn(f"Terminal file not found: {terminals_path}")
        terminal_features = pd.DataFrame(columns=["terminal_id", "assigned_faf", "inside_region", "access_distance_miles", "x_proj", "y_proj"])
        access_edges = pd.DataFrame(columns=["src_faf", "terminal_id", "access_distance_miles", "inside_region"])
        return terminal_features, access_edges

    terminals = gpd.read_parquet(terminals_path)
    terminals = ensure_geodataframe(terminals, LON_CANDIDATES, LAT_CANDIDATES)
    terminals = terminals.dropna(subset=[terminals.geometry.name]).copy()
    terminals_projected = project_gdf(terminals, projected_crs)
    regions_projected = project_gdf(faf_regions_gdf[["faf_zone_str", "geometry"]], projected_crs)

    terminal_id_col = find_first_existing_column(terminals_projected.columns, TERMINAL_ID_CANDIDATES, required=False)
    if terminal_id_col is None:
        terminals_projected["terminal_id"] = [f"terminal_{idx:04d}" for idx in range(len(terminals_projected))]
    else:
        terminals_projected["terminal_id"] = terminals_projected[terminal_id_col].astype(str).fillna("")
        terminals_projected.loc[terminals_projected["terminal_id"].eq(""), "terminal_id"] = [
            f"terminal_{idx:04d}" for idx in range(len(terminals_projected[terminals_projected["terminal_id"].eq("")]))
        ]

    terminals_projected = terminals_projected.reset_index(drop=True)
    terminals_projected["terminal_row_id"] = np.arange(len(terminals_projected), dtype=np.int64)

    inside = gpd.sjoin(
        terminals_projected[["terminal_row_id", "terminal_id", "geometry"]],
        regions_projected[["faf_zone_str", "geometry"]],
        how="left",
        predicate="within",
    )
    inside_assignment = inside.dropna(subset=["faf_zone_str"]).drop_duplicates("terminal_row_id")[["terminal_row_id", "faf_zone_str"]]
    inside_assignment = inside_assignment.rename(columns={"faf_zone_str": "inside_faf"})

    terminals_projected = terminals_projected.merge(inside_assignment, on="terminal_row_id", how="left")
    terminals_projected["inside_region"] = terminals_projected["inside_faf"].notna()

    missing = terminals_projected[terminals_projected["inside_faf"].isna()].copy()
    nearest_assignment = pd.DataFrame(columns=["terminal_row_id", "nearest_faf", "nearest_distance_miles"])
    if not missing.empty:
        nearest = gpd.sjoin_nearest(
            missing[["terminal_row_id", "terminal_id", "geometry"]],
            regions_projected[["faf_zone_str", "geometry"]],
            how="left",
            distance_col="distance_meters",
        )
        nearest_assignment = nearest.drop_duplicates("terminal_row_id")[["terminal_row_id", "faf_zone_str", "distance_meters"]].copy()
        nearest_assignment = nearest_assignment.rename(columns={"faf_zone_str": "nearest_faf"})
        nearest_assignment["nearest_distance_miles"] = nearest_assignment["distance_meters"].astype(float) / 1609.344
        nearest_assignment = nearest_assignment.drop(columns=["distance_meters"])

    terminals_projected = terminals_projected.merge(nearest_assignment, on="terminal_row_id", how="left")
    terminals_projected["assigned_faf"] = terminals_projected["inside_faf"].fillna(terminals_projected["nearest_faf"]).map(normalize_faf_code)
    terminals_projected["access_distance_miles"] = terminals_projected["nearest_distance_miles"].fillna(0.0)

    terminal_features = pd.DataFrame(
        {
            "terminal_id": terminals_projected["terminal_id"].astype(str),
            "assigned_faf": terminals_projected["assigned_faf"],
            "inside_region": terminals_projected["inside_region"].astype(bool),
            "access_distance_miles": terminals_projected["access_distance_miles"].astype(float),
            "x_proj": terminals_projected.geometry.x.astype(float),
            "y_proj": terminals_projected.geometry.y.astype(float),
        }
    )
    terminal_features = terminal_features.dropna(subset=["assigned_faf"]).reset_index(drop=True)

    access_edges = terminal_features[["assigned_faf", "terminal_id", "access_distance_miles", "inside_region"]].rename(columns={"assigned_faf": "src_faf"})
    return terminal_features, access_edges


terminal_features, terminal_access_edges = build_terminal_assignments(paths.terminals_path, faf_regions, cfg.projected_crs)
terminal_ids = sorted(terminal_features["terminal_id"].astype(str).unique())
terminal_to_idx = {terminal_id: idx for idx, terminal_id in enumerate(terminal_ids)}

print("Terminal features:", terminal_features.shape)
print("Terminal access edges:", terminal_access_edges.shape)
log_frame("Terminal assignments preview", terminal_features)

Terminal features: (241, 6)
Terminal access edges: (241, 4)

Terminal assignments preview: shape=(241, 6)
terminal_id assigned_faf  inside_region  access_distance_miles        x_proj       y_proj
          1          531           True                    0.0 -2.026430e+06 2.962184e+06
          2          411           True                    0.0 -2.072848e+06 2.874132e+06
          3          061           True                    0.0 -2.102317e+06 1.489422e+06
          4          489           True                    0.0 -5.422695e+04 6.195697e+05
          5          012           True                    0.0  7.600027e+05 8.755568e+05


C:\Users\Nick_James\AppData\Local\Temp\ipykernel_4324\808828658.py:60: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  terminals_projected["access_distance_miles"] = terminals_projected["nearest_distance_miles"].fillna(0.0)


## 10. Build truck and rail FAF-zone adjacency edges

This cell constructs coarse zone-level truck and rail topology. It uses chunked spatial joins between line geometries and FAF region polygons.

For each physical line that intersects more than one FAF zone, we create undirected zone-to-zone adjacency. Edge distance is an aggregate of line geometry lengths and is later converted to directed PyG edges.

In [14]:
# ---------------------------------------------------------------------
# Build truck and rail FAF-zone adjacency edges with safe string handling
# ---------------------------------------------------------------------
#
# This cell replaces the original truck/rail adjacency construction cell.
#
# Why this replacement is needed:
# The original aggregate_undirected_edges() used np.minimum / np.maximum
# on string arrays to canonicalize undirected FAF-zone pairs. In some
# NumPy versions, string ufunc comparison fails with:
#
#   UFuncTypeError: ufunc 'minimum' did not contain a loop with signature
#   matching types (dtype('<U3'), dtype('<U3')) -> None
#
# This cell redefines aggregate_undirected_edges() using pure Python
# pair canonicalization, which is robust for string FAF codes such as
# "011", "012", "089".

from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import pandas as pd


def normalize_faf_code_for_topology_edges(value: object) -> Optional[str]:
    """
    Normalize a FAF-zone identifier into a 3-character string.

    This helper first tries the project's existing normalize_faf_code()
    function. If that fails or returns an unusable value, it falls back
    to a simple digit-based normalization.
    """
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    # Prefer the existing project normalizer when available.
    try:
        code = normalize_faf_code(value)
        if code is not None:
            code_text = str(code).strip()
            if code_text and code_text.lower() not in {"nan", "none"}:
                digits = "".join(ch for ch in code_text if ch.isdigit())
                if digits:
                    return digits[-3:].zfill(3)
    except Exception:
        pass

    text = str(value).strip()
    if not text:
        return None

    digits = "".join(ch for ch in text if ch.isdigit())
    if not digits:
        return None

    return digits[-3:].zfill(3)


def canonicalize_undirected_faf_pair(src: object, dst: object) -> Optional[Tuple[str, str]]:
    """
    Return a canonical undirected FAF-zone pair.

    The smaller string code is always placed first. This avoids using
    np.minimum / np.maximum on string arrays.
    """
    src_code = normalize_faf_code_for_topology_edges(src)
    dst_code = normalize_faf_code_for_topology_edges(dst)

    if src_code is None or dst_code is None:
        return None

    if src_code == dst_code:
        return None

    if src_code <= dst_code:
        return src_code, dst_code

    return dst_code, src_code


def empty_adjacency_frame() -> pd.DataFrame:
    """Return an empty adjacency table with the expected downstream schema."""
    return pd.DataFrame(
        columns=[
            "src_faf",
            "dst_faf",
            "source_relation",
            "distance_miles",
            "min_distance_miles",
            "edge_count",
        ]
    )


def aggregate_undirected_edges(edge_rows: List[Tuple[str, str, float, str]]) -> pd.DataFrame:
    """
    Aggregate raw zone-zone line intersections into undirected FAF adjacency edges.

    Input rows have schema:
        (src_faf, dst_faf, distance_miles, source_relation)

    Output rows have schema:
        src_faf
        dst_faf
        source_relation
        distance_miles
        min_distance_miles
        edge_count
    """
    if not edge_rows:
        return empty_adjacency_frame()

    raw = pd.DataFrame(
        edge_rows,
        columns=["src_faf", "dst_faf", "distance_miles", "source_relation"],
    )

    if raw.empty:
        return empty_adjacency_frame()

    # Normalize relation and distance fields.
    raw["source_relation"] = raw["source_relation"].astype("string")
    raw["distance_miles"] = pd.to_numeric(raw["distance_miles"], errors="coerce")
    raw["distance_miles"] = raw["distance_miles"].replace([np.inf, -np.inf], np.nan)

    # Canonicalize undirected pairs using pure Python string comparison.
    canonical_pairs = [
        canonicalize_undirected_faf_pair(src, dst)
        for src, dst in zip(raw["src_faf"], raw["dst_faf"])
    ]

    valid_mask = [pair is not None for pair in canonical_pairs]
    if not any(valid_mask):
        return empty_adjacency_frame()

    raw = raw.loc[valid_mask].copy()
    canonical_pairs = [pair for pair in canonical_pairs if pair is not None]

    raw["src_faf"] = [pair[0] for pair in canonical_pairs]
    raw["dst_faf"] = [pair[1] for pair in canonical_pairs]

    # Keep only valid relation names.
    raw = raw.dropna(subset=["src_faf", "dst_faf", "source_relation"]).copy()
    if raw.empty:
        return empty_adjacency_frame()

    # Aggregate repeated line intersections between the same FAF-zone pair.
    aggregated = (
        raw.groupby(["src_faf", "dst_faf", "source_relation"], as_index=False)
        .agg(
            distance_miles=("distance_miles", "median"),
            min_distance_miles=("distance_miles", "min"),
            edge_count=("distance_miles", "size"),
        )
    )

    # Ensure stable dtypes for downstream parquet and graph construction.
    aggregated["src_faf"] = aggregated["src_faf"].astype("string")
    aggregated["dst_faf"] = aggregated["dst_faf"].astype("string")
    aggregated["source_relation"] = aggregated["source_relation"].astype("string")
    aggregated["distance_miles"] = pd.to_numeric(aggregated["distance_miles"], errors="coerce")
    aggregated["min_distance_miles"] = pd.to_numeric(aggregated["min_distance_miles"], errors="coerce")
    aggregated["edge_count"] = pd.to_numeric(
        aggregated["edge_count"],
        errors="coerce",
    ).fillna(0).astype(np.int64)

    return aggregated[
        [
            "src_faf",
            "dst_faf",
            "source_relation",
            "distance_miles",
            "min_distance_miles",
            "edge_count",
        ]
    ].reset_index(drop=True)


def validate_adjacency_table(table: pd.DataFrame, relation_name: str) -> pd.DataFrame:
    """
    Validate and normalize a truck_adj or rail_adj adjacency table.

    This keeps the rest of the notebook safer if a cached file was created
    by an earlier version of the code.
    """
    expected_columns = [
        "src_faf",
        "dst_faf",
        "source_relation",
        "distance_miles",
        "min_distance_miles",
        "edge_count",
    ]

    if table is None or table.empty:
        return empty_adjacency_frame()

    clean = table.copy()

    # Backward-compatible handling if a cached table has slightly different names.
    rename_map = {}
    if "relation" in clean.columns and "source_relation" not in clean.columns:
        rename_map["relation"] = "source_relation"
    if rename_map:
        clean = clean.rename(columns=rename_map)

    for column in expected_columns:
        if column not in clean.columns:
            if column == "source_relation":
                clean[column] = relation_name
            elif column in {"distance_miles", "min_distance_miles"}:
                clean[column] = np.nan
            elif column == "edge_count":
                clean[column] = 1
            else:
                clean[column] = pd.NA

    clean["src_faf"] = clean["src_faf"].map(normalize_faf_code_for_topology_edges)
    clean["dst_faf"] = clean["dst_faf"].map(normalize_faf_code_for_topology_edges)
    clean["source_relation"] = clean["source_relation"].fillna(relation_name).astype("string")

    clean = clean.dropna(subset=["src_faf", "dst_faf"]).copy()
    clean = clean[clean["src_faf"].ne(clean["dst_faf"])].copy()

    if clean.empty:
        return empty_adjacency_frame()

    # Re-canonicalize cached tables as well.
    canonical_pairs = [
        canonicalize_undirected_faf_pair(src, dst)
        for src, dst in zip(clean["src_faf"], clean["dst_faf"])
    ]
    valid_mask = [pair is not None for pair in canonical_pairs]
    clean = clean.loc[valid_mask].copy()
    canonical_pairs = [pair for pair in canonical_pairs if pair is not None]

    clean["src_faf"] = [pair[0] for pair in canonical_pairs]
    clean["dst_faf"] = [pair[1] for pair in canonical_pairs]

    clean["distance_miles"] = pd.to_numeric(clean["distance_miles"], errors="coerce")
    clean["min_distance_miles"] = pd.to_numeric(clean["min_distance_miles"], errors="coerce")
    clean["edge_count"] = pd.to_numeric(clean["edge_count"], errors="coerce").fillna(1).astype(np.int64)

    # If the cache had duplicated pairs, aggregate again.
    clean = (
        clean.groupby(["src_faf", "dst_faf", "source_relation"], as_index=False)
        .agg(
            distance_miles=("distance_miles", "median"),
            min_distance_miles=("min_distance_miles", "min"),
            edge_count=("edge_count", "sum"),
        )
    )

    return clean[expected_columns].reset_index(drop=True)


# ---------------------------------------------------------------------
# Build or load adjacency edges.
# ---------------------------------------------------------------------

truck_adj_cache = paths.cache_dir / "truck_adj_edges.parquet"
rail_adj_cache = paths.cache_dir / "rail_adj_edges.parquet"

truck_adj_edges = build_zone_adjacency_from_line_chunks(
    paths.truck_links_dir,
    faf_regions,
    relation_name="truck_adj",
    cache_path=truck_adj_cache,
    max_files=cfg.max_truck_topology_files,
    reuse_cache=cfg.reuse_cached_topology_edges,
    predicate=cfg.topology_predicate,
    projected_crs=cfg.projected_crs,
)

truck_adj_edges = validate_adjacency_table(truck_adj_edges, relation_name="truck_adj")

# Save the normalized table so future reruns can load a clean cache.
if not truck_adj_edges.empty:
    safe_to_parquet(truck_adj_edges, truck_adj_cache)


rail_adj_edges = build_zone_adjacency_from_line_chunks(
    paths.rail_lines_dir,
    faf_regions,
    relation_name="rail_adj",
    cache_path=rail_adj_cache,
    max_files=cfg.max_rail_topology_files,
    reuse_cache=cfg.reuse_cached_topology_edges,
    predicate=cfg.topology_predicate,
    projected_crs=cfg.projected_crs,
)

rail_adj_edges = validate_adjacency_table(rail_adj_edges, relation_name="rail_adj")

# Save the normalized table so future reruns can load a clean cache.
if not rail_adj_edges.empty:
    safe_to_parquet(rail_adj_edges, rail_adj_cache)


# ---------------------------------------------------------------------
# Diagnostics.
# ---------------------------------------------------------------------

print("Truck adjacency edges:", truck_adj_edges.shape)
print("Rail adjacency edges:", rail_adj_edges.shape)

if truck_adj_edges.empty:
    print(
        "Warning: truck_adj_edges is empty. Check FAF region geometries, "
        "truck link geometries, CRS, and spatial predicate."
    )

if rail_adj_edges.empty:
    print(
        "Warning: rail_adj_edges is empty. Check FAF region geometries, "
        "rail line geometries, CRS, and spatial predicate."
    )

log_frame("Truck adjacency preview", truck_adj_edges.head(20))
log_frame("Rail adjacency preview", rail_adj_edges.head(20))

[truck_adj] processing 1/488: faf_network_links__chunk_00001.parquet
[truck_adj] processing 2/488: faf_network_links__chunk_00002.parquet
[truck_adj] processing 3/488: faf_network_links__chunk_00003.parquet
[truck_adj] processing 4/488: faf_network_links__chunk_00004.parquet
[truck_adj] processing 5/488: faf_network_links__chunk_00005.parquet
[truck_adj] processing 6/488: faf_network_links__chunk_00006.parquet
[truck_adj] processing 7/488: faf_network_links__chunk_00007.parquet
[truck_adj] processing 8/488: faf_network_links__chunk_00008.parquet
[truck_adj] processing 9/488: faf_network_links__chunk_00009.parquet
[truck_adj] processing 10/488: faf_network_links__chunk_00010.parquet
[truck_adj] processing 11/488: faf_network_links__chunk_00011.parquet
[truck_adj] processing 12/488: faf_network_links__chunk_00012.parquet
[truck_adj] processing 13/488: faf_network_links__chunk_00013.parquet
[truck_adj] processing 14/488: faf_network_links__chunk_00014.parquet
[truck_adj] processing 15/488

## 11. Build shortest-path and terminal topology features for each OD-year row

The edge decoder will receive explicit topology features, including truck/rail shortest path length, truck-to-rail path ratio, terminal availability, and terminal access distance.

Unreachable paths are kept with missing flags and imputed values. This avoids silently treating missing topology as zero distance.

In [15]:
def build_weighted_graph(edge_df: pd.DataFrame, nodes: Sequence[str]) -> nx.Graph:
    """Build an undirected weighted NetworkX graph from zone-zone topology edges."""
    graph = nx.Graph()
    graph.add_nodes_from(nodes)
    if edge_df.empty:
        return graph

    for row in edge_df.itertuples(index=False):
        src = normalize_faf_code(getattr(row, "src_faf"))
        dst = normalize_faf_code(getattr(row, "dst_faf"))
        if src is None or dst is None or src == dst:
            continue
        distance = float(getattr(row, "distance_miles", np.nan))
        if not np.isfinite(distance) or distance <= 0:
            distance = 1.0
        if graph.has_edge(src, dst):
            graph[src][dst]["distance_miles"] = min(graph[src][dst]["distance_miles"], distance)
            graph[src][dst]["edge_count"] += int(getattr(row, "edge_count", 1))
        else:
            graph.add_edge(src, dst, distance_miles=distance, edge_count=int(getattr(row, "edge_count", 1)))
    return graph


def compute_all_pairs_lengths(graph: nx.Graph) -> Dict[Tuple[str, str], float]:
    """Compute all-pairs shortest path lengths for a small FAF-zone graph."""
    lengths: Dict[Tuple[str, str], float] = {}
    if graph.number_of_edges() == 0:
        return lengths
    for src, dst_lengths in nx.all_pairs_dijkstra_path_length(graph, weight="distance_miles"):
        for dst, distance in dst_lengths.items():
            lengths[(src, dst)] = float(distance)
    return lengths


def lookup_distance(lengths: Dict[Tuple[str, str], float], src: str, dst: str) -> float:
    """Return a shortest-path distance or NaN if the path is unavailable."""
    src = normalize_faf_code(src)
    dst = normalize_faf_code(dst)
    if src is None or dst is None:
        return np.nan
    if src == dst:
        return 0.0
    return lengths.get((src, dst), np.nan)


def build_zone_terminal_features(zones: Sequence[str], terminal_features_df: pd.DataFrame, default_distance_miles: float) -> pd.DataFrame:
    """Build per-FAF-zone terminal count and nearest-terminal features."""
    rows = []
    for zone in zones:
        subset = terminal_features_df[terminal_features_df["assigned_faf"].map(normalize_faf_code).eq(zone)]
        if subset.empty:
            rows.append(
                {
                    "faf_zone_str": zone,
                    "terminal_count": 0,
                    "nearest_terminal_distance_miles_raw": np.nan,
                    "nearest_terminal_distance_miles": default_distance_miles,
                    "has_terminal": 0,
                }
            )
        else:
            nearest = float(subset["access_distance_miles"].min())
            rows.append(
                {
                    "faf_zone_str": zone,
                    "terminal_count": int(len(subset)),
                    "nearest_terminal_distance_miles_raw": nearest,
                    "nearest_terminal_distance_miles": nearest,
                    "has_terminal": 1,
                }
            )
    return pd.DataFrame(rows)


def impute_distance_column(values: pd.Series, train_mask: pd.Series, strategy: str) -> Tuple[pd.Series, float]:
    """Impute missing distance values using train median or zero."""
    numeric = pd.to_numeric(values, errors="coerce")
    if strategy == "zero":
        fill_value = 0.0
    elif strategy == "train_median":
        train_values = numeric[train_mask & numeric.notna()]
        fill_value = float(train_values.median()) if not train_values.empty else 0.0
    else:
        raise ValueError(f"Unsupported distance fill strategy: {strategy}")
    return numeric.fillna(fill_value), fill_value


truck_graph = build_weighted_graph(truck_adj_edges, all_faf_zones)
rail_graph = build_weighted_graph(rail_adj_edges, all_faf_zones)
truck_lengths = compute_all_pairs_lengths(truck_graph)
rail_lengths = compute_all_pairs_lengths(rail_graph)

zone_terminal_features = build_zone_terminal_features(
    all_faf_zones,
    terminal_features,
    default_distance_miles=cfg.terminal_nearest_distance_fill_miles,
)

origin_terminal = zone_terminal_features.add_prefix("orig_").rename(columns={"orig_faf_zone_str": "faf_orig_str"})
dest_terminal = zone_terminal_features.add_prefix("dest_").rename(columns={"dest_faf_zone_str": "faf_dest_str"})

topology_features = supervised_df[["faf_orig_str", "faf_dest_str", "year_int", "temporal_split", "cold_split"]].copy()
topology_features["topo_truck_distance_miles_raw"] = [
    lookup_distance(truck_lengths, src, dst) for src, dst in zip(topology_features["faf_orig_str"], topology_features["faf_dest_str"])
]
topology_features["topo_rail_distance_miles_raw"] = [
    lookup_distance(rail_lengths, src, dst) for src, dst in zip(topology_features["faf_orig_str"], topology_features["faf_dest_str"])
]

temporal_train_mask = topology_features["temporal_split"].eq("train")
topology_features["topo_truck_path_available"] = topology_features["topo_truck_distance_miles_raw"].notna().astype(np.float32)
topology_features["topo_rail_path_available"] = topology_features["topo_rail_distance_miles_raw"].notna().astype(np.float32)

topology_features["topo_truck_distance_miles"], truck_fill = impute_distance_column(
    topology_features["topo_truck_distance_miles_raw"], temporal_train_mask, cfg.unreachable_distance_fill_strategy
)
topology_features["topo_rail_distance_miles"], rail_fill = impute_distance_column(
    topology_features["topo_rail_distance_miles_raw"], temporal_train_mask, cfg.unreachable_distance_fill_strategy
)

topology_features["topo_log1p_truck_distance_miles"] = np.log1p(topology_features["topo_truck_distance_miles"].clip(lower=0))
topology_features["topo_log1p_rail_distance_miles"] = np.log1p(topology_features["topo_rail_distance_miles"].clip(lower=0))
topology_features["topo_truck_rail_distance_ratio"] = (
    topology_features["topo_truck_distance_miles"] / topology_features["topo_rail_distance_miles"].replace(0, np.nan)
).replace([np.inf, -np.inf], np.nan)
ratio_fill = topology_features.loc[temporal_train_mask, "topo_truck_rail_distance_ratio"].median()
topology_features["topo_truck_rail_distance_ratio"] = topology_features["topo_truck_rail_distance_ratio"].fillna(float(ratio_fill) if np.isfinite(ratio_fill) else 1.0)

topology_features = topology_features.merge(origin_terminal, on="faf_orig_str", how="left")
topology_features = topology_features.merge(dest_terminal, on="faf_dest_str", how="left")

for column in [
    "orig_terminal_count",
    "orig_nearest_terminal_distance_miles",
    "orig_has_terminal",
    "dest_terminal_count",
    "dest_nearest_terminal_distance_miles",
    "dest_has_terminal",
]:
    if column not in topology_features.columns:
        topology_features[column] = 0
    topology_features[column] = pd.to_numeric(topology_features[column], errors="coerce").fillna(0)

topology_features["topo_origin_terminal_count"] = topology_features["orig_terminal_count"].astype(float)
topology_features["topo_dest_terminal_count"] = topology_features["dest_terminal_count"].astype(float)
topology_features["topo_origin_nearest_terminal_distance_miles"] = topology_features["orig_nearest_terminal_distance_miles"].astype(float)
topology_features["topo_dest_nearest_terminal_distance_miles"] = topology_features["dest_nearest_terminal_distance_miles"].astype(float)
topology_features["topo_log1p_origin_nearest_terminal_distance_miles"] = np.log1p(topology_features["topo_origin_nearest_terminal_distance_miles"].clip(lower=0))
topology_features["topo_log1p_dest_nearest_terminal_distance_miles"] = np.log1p(topology_features["topo_dest_nearest_terminal_distance_miles"].clip(lower=0))
topology_features["topo_both_zones_have_terminal"] = (
    topology_features["orig_has_terminal"].gt(0) & topology_features["dest_has_terminal"].gt(0)
).astype(np.float32)
topology_features["topo_terminal_rail_path_available"] = (
    topology_features["topo_both_zones_have_terminal"].gt(0) & topology_features["topo_rail_path_available"].gt(0)
).astype(np.float32)
topology_features["topo_same_faf_zone"] = topology_features["faf_orig_str"].eq(topology_features["faf_dest_str"]).astype(np.float32)

topology_feature_columns = [
    "topo_truck_distance_miles",
    "topo_rail_distance_miles",
    "topo_log1p_truck_distance_miles",
    "topo_log1p_rail_distance_miles",
    "topo_truck_path_available",
    "topo_rail_path_available",
    "topo_truck_rail_distance_ratio",
    "topo_origin_terminal_count",
    "topo_dest_terminal_count",
    "topo_origin_nearest_terminal_distance_miles",
    "topo_dest_nearest_terminal_distance_miles",
    "topo_log1p_origin_nearest_terminal_distance_miles",
    "topo_log1p_dest_nearest_terminal_distance_miles",
    "topo_both_zones_have_terminal",
    "topo_terminal_rail_path_available",
    "topo_same_faf_zone",
]

# Add row identity columns for downstream merges.
topology_features.insert(0, "row_id", np.arange(len(topology_features), dtype=np.int64))

print("Truck graph nodes/edges:", truck_graph.number_of_nodes(), truck_graph.number_of_edges())
print("Rail graph nodes/edges:", rail_graph.number_of_nodes(), rail_graph.number_of_edges())
print("Truck distance fill:", truck_fill, "Rail distance fill:", rail_fill)
log_frame("Topology feature preview", topology_features[["row_id", "faf_orig_str", "faf_dest_str"] + topology_feature_columns])

Truck graph nodes/edges: 132 291
Rail graph nodes/edges: 132 255
Truck distance fill: 6.946667202281741 Rail distance fill: 4.369057100549882

Topology feature preview: shape=(73972, 19)
 row_id faf_orig_str faf_dest_str  topo_truck_distance_miles  topo_rail_distance_miles  topo_log1p_truck_distance_miles  topo_log1p_rail_distance_miles  topo_truck_path_available  topo_rail_path_available  topo_truck_rail_distance_ratio  topo_origin_terminal_count  topo_dest_terminal_count  topo_origin_nearest_terminal_distance_miles  topo_dest_nearest_terminal_distance_miles  topo_log1p_origin_nearest_terminal_distance_miles  topo_log1p_dest_nearest_terminal_distance_miles  topo_both_zones_have_terminal  topo_terminal_rail_path_available  topo_same_faf_zone
      0          011          011                   0.000000                  0.000000                         0.000000                        0.000000                        1.0                       1.0                        1.483878            

## 12. Build train-only FAF node features

FAF node features are computed from training rows only to avoid using validation/test label-derived structure. These features summarize outgoing and incoming freight demand, train-row counts, topology degree, and terminal availability.

In [16]:
DEMAND_BASE_COLUMNS = [
    "tons_truck", "tons_rail", "tons_multimodal",
    "tmiles_truck", "tmiles_rail", "tmiles_multimodal",
    "value_truck", "value_rail", "value_multimodal",
    "total_tons_modes_1_2_5", "total_tmiles_modes_1_2_5", "total_value_modes_1_2_5",
]
DEMAND_BASE_COLUMNS = [column for column in DEMAND_BASE_COLUMNS if column in supervised_df.columns]


def aggregate_zone_role_features(frame: pd.DataFrame, zone_column: str, prefix: str, demand_columns: Sequence[str]) -> pd.DataFrame:
    """Aggregate demand features by FAF zone for origin or destination roles."""
    if frame.empty:
        return pd.DataFrame({"faf_zone_str": []})

    aggregations = {column: ["sum", "mean"] for column in demand_columns}
    grouped = frame.groupby(zone_column).agg(aggregations)
    grouped.columns = [f"{prefix}_{column}_{stat}" for column, stat in grouped.columns]
    grouped = grouped.reset_index().rename(columns={zone_column: "faf_zone_str"})
    grouped[f"{prefix}_train_rows"] = frame.groupby(zone_column).size().reindex(grouped["faf_zone_str"]).to_numpy()
    return grouped


def topology_degree_features(zones: Sequence[str], truck_edges: pd.DataFrame, rail_edges: pd.DataFrame) -> pd.DataFrame:
    """Compute simple truck/rail degree features from topology edge tables."""
    rows = []
    for zone in zones:
        truck_degree = 0
        rail_degree = 0
        if not truck_edges.empty:
            truck_degree = int((truck_edges["src_faf"].eq(zone) | truck_edges["dst_faf"].eq(zone)).sum())
        if not rail_edges.empty:
            rail_degree = int((rail_edges["src_faf"].eq(zone) | rail_edges["dst_faf"].eq(zone)).sum())
        rows.append({"faf_zone_str": zone, "truck_topology_degree": truck_degree, "rail_topology_degree": rail_degree})
    return pd.DataFrame(rows)


node_train_mask = supervised_df["cold_split"].eq("train") if supervised_df["cold_split"].eq("train").any() else supervised_df["temporal_split"].eq("train")
node_train_df = supervised_df.loc[node_train_mask].copy()

outgoing_features = aggregate_zone_role_features(node_train_df, "faf_orig_str", "out", DEMAND_BASE_COLUMNS)
incoming_features = aggregate_zone_role_features(node_train_df, "faf_dest_str", "in", DEMAND_BASE_COLUMNS)
degree_features = topology_degree_features(all_faf_zones, truck_adj_edges, rail_adj_edges)

faf_node_feature_df = pd.DataFrame({"faf_zone_str": all_faf_zones})
faf_node_feature_df["faf_zone_numeric"] = faf_node_feature_df["faf_zone_str"].astype(float)
faf_node_feature_df = faf_node_feature_df.merge(outgoing_features, on="faf_zone_str", how="left")
faf_node_feature_df = faf_node_feature_df.merge(incoming_features, on="faf_zone_str", how="left")
faf_node_feature_df = faf_node_feature_df.merge(degree_features, on="faf_zone_str", how="left")
faf_node_feature_df = faf_node_feature_df.merge(zone_terminal_features, on="faf_zone_str", how="left")

for column in faf_node_feature_df.columns:
    if column != "faf_zone_str":
        faf_node_feature_df[column] = pd.to_numeric(faf_node_feature_df[column], errors="coerce").fillna(0.0)

# Log-transform count and sum columns while keeping means and flags raw.
for column in list(faf_node_feature_df.columns):
    if column == "faf_zone_str":
        continue
    if any(token in column for token in ["_sum", "_rows", "terminal_count", "topology_degree"]):
        faf_node_feature_df[f"log1p_{column}"] = np.log1p(faf_node_feature_df[column].clip(lower=0))

node_feature_columns = [column for column in faf_node_feature_df.columns if column != "faf_zone_str"]

# Standardize FAF node features.
node_values = faf_node_feature_df[node_feature_columns].astype(float).to_numpy(dtype=np.float32)
node_mean = node_values.mean(axis=0)
node_std = node_values.std(axis=0)
node_std[node_std < 1e-6] = 1.0
node_x = (node_values - node_mean) / node_std

print("FAF node feature shape:", node_x.shape)
log_frame("FAF node features preview", faf_node_feature_df[["faf_zone_str"] + node_feature_columns[:12]])

FAF node feature shape: (132, 86)

FAF node features preview: shape=(132, 13)
faf_zone_str  faf_zone_numeric  out_tons_truck_sum  out_tons_truck_mean  out_tons_rail_sum  out_tons_rail_mean  out_tons_multimodal_sum  out_tons_multimodal_mean  out_tmiles_truck_sum  out_tmiles_truck_mean  out_tmiles_rail_sum  out_tmiles_rail_mean  out_tmiles_multimodal_sum
         011              11.0       313905.891614           760.062692       23351.891899           56.542111             22417.512403                 54.279691          41782.967528             101.169413          8326.557623             20.161156               11753.247749
         012              12.0       121082.963507           308.885111       15628.710083           39.869158             10033.433189                 25.595493          18922.856319              48.272593          7123.248469             18.171552                6826.773210
         019              19.0       555916.204367          1372.632603       72451.644007 

## 13. Build terminal node features

Terminal features are intentionally simple: projected coordinates, access distance, inside-zone flag, and assigned FAF zone code. The coordinates are standardized before being stored in the PyG object.

In [17]:
def build_terminal_node_features(terminal_features_df: pd.DataFrame) -> Tuple[pd.DataFrame, np.ndarray, List[str], np.ndarray, np.ndarray]:
    """Create standardized terminal node features."""
    if terminal_features_df.empty:
        terminal_df = pd.DataFrame({"terminal_id": []})
        return terminal_df, np.zeros((0, 1), dtype=np.float32), ["constant_feature"], np.zeros(1, dtype=np.float32), np.ones(1, dtype=np.float32)

    terminal_df = terminal_features_df.copy()
    terminal_df["assigned_faf_numeric"] = terminal_df["assigned_faf"].map(lambda x: float(normalize_faf_code(x) or 0))
    terminal_df["inside_region_float"] = terminal_df["inside_region"].astype(float)
    terminal_df["log1p_access_distance_miles"] = np.log1p(pd.to_numeric(terminal_df["access_distance_miles"], errors="coerce").fillna(0).clip(lower=0))

    columns = ["x_proj", "y_proj", "assigned_faf_numeric", "inside_region_float", "log1p_access_distance_miles"]
    values = terminal_df[columns].astype(float).to_numpy(dtype=np.float32)
    mean = values.mean(axis=0)
    std = values.std(axis=0)
    std[std < 1e-6] = 1.0
    x = (values - mean) / std
    return terminal_df, x, columns, mean, std


terminal_node_df, terminal_x, terminal_feature_columns, terminal_mean, terminal_std = build_terminal_node_features(terminal_features)
print("Terminal node feature shape:", terminal_x.shape)
log_frame("Terminal node feature preview", terminal_node_df.head())

Terminal node feature shape: (241, 5)

Terminal node feature preview: shape=(5, 9)
terminal_id assigned_faf  inside_region  access_distance_miles        x_proj       y_proj  assigned_faf_numeric  inside_region_float  log1p_access_distance_miles
          1          531           True                    0.0 -2.026430e+06 2.962184e+06                 531.0                  1.0                          0.0
          2          411           True                    0.0 -2.072848e+06 2.874132e+06                 411.0                  1.0                          0.0
          3          061           True                    0.0 -2.102317e+06 1.489422e+06                  61.0                  1.0                          0.0
          4          489           True                    0.0 -5.422695e+04 6.195697e+05                 489.0                  1.0                          0.0
          5          012           True                    0.0  7.600027e+05 8.755568e+05                  

## 14. Build edge index utilities and typed graph edges

This cell converts zone/terminal edge tables into PyG-style `edge_index` tensors. It also creates train-only demand edges and train-only supervised OD message edges. The actual supervised labels are stored separately as `edge_label_index` and are not used as message edges for validation/test rows.

In [18]:
def directed_edge_index_from_pairs(pairs: Iterable[Tuple[str, str]], node_map: Dict[str, int]) -> torch.Tensor:
    """Convert directed FAF pair tuples into a PyG edge_index tensor."""
    src_indices: List[int] = []
    dst_indices: List[int] = []
    for src, dst in pairs:
        src = normalize_faf_code(src)
        dst = normalize_faf_code(dst)
        if src in node_map and dst in node_map:
            src_indices.append(node_map[src])
            dst_indices.append(node_map[dst])
    if not src_indices:
        return torch.empty((2, 0), dtype=torch.long)
    return torch.tensor([src_indices, dst_indices], dtype=torch.long)


def topology_edge_index(edge_df: pd.DataFrame, node_map: Dict[str, int], make_bidirectional: bool = True) -> Tuple[torch.Tensor, torch.Tensor]:
    """Convert undirected topology edge table to directed edge_index and edge_attr."""
    pairs: List[Tuple[str, str]] = []
    attrs: List[List[float]] = []
    for row in edge_df.itertuples(index=False):
        src = normalize_faf_code(getattr(row, "src_faf"))
        dst = normalize_faf_code(getattr(row, "dst_faf"))
        if src not in node_map or dst not in node_map or src == dst:
            continue
        distance = float(getattr(row, "distance_miles", 0.0))
        edge_count = float(getattr(row, "edge_count", 1.0))
        attr = [distance, math.log1p(max(distance, 0.0)), edge_count]
        pairs.append((src, dst))
        attrs.append(attr)
        if make_bidirectional:
            pairs.append((dst, src))
            attrs.append(attr)
    edge_index = directed_edge_index_from_pairs(pairs, node_map)
    edge_attr = torch.tensor(attrs, dtype=torch.float32) if attrs else torch.zeros((0, 3), dtype=torch.float32)
    return edge_index, edge_attr


def demand_edge_table(frame: pd.DataFrame, mode: str, train_mask: pd.Series) -> pd.DataFrame:
    """Build train-only mode-specific demand edge table from supervised OD rows."""
    mode_to_tons = {
        "truck": "tons_truck",
        "rail": "tons_rail",
        "multimodal": "tons_multimodal",
    }
    tons_col = mode_to_tons[mode]
    if tons_col not in frame.columns:
        return pd.DataFrame(columns=["src_faf", "dst_faf", "edge_count", "tons_sum", "tmiles_sum", "value_sum"])

    tmiles_col = f"tmiles_{mode}" if f"tmiles_{mode}" in frame.columns else None
    value_col = f"value_{mode}" if f"value_{mode}" in frame.columns else None

    subset = frame.loc[train_mask & pd.to_numeric(frame[tons_col], errors="coerce").fillna(0).gt(0)].copy()
    if subset.empty:
        return pd.DataFrame(columns=["src_faf", "dst_faf", "edge_count", "tons_sum", "tmiles_sum", "value_sum"])

    subset["tmiles_for_mode"] = pd.to_numeric(subset[tmiles_col], errors="coerce").fillna(0.0) if tmiles_col else 0.0
    subset["value_for_mode"] = pd.to_numeric(subset[value_col], errors="coerce").fillna(0.0) if value_col else 0.0
    grouped = (
        subset.groupby(["faf_orig_str", "faf_dest_str"], as_index=False)
        .agg(edge_count=(tons_col, "size"), tons_sum=(tons_col, "sum"), tmiles_sum=("tmiles_for_mode", "sum"), value_sum=("value_for_mode", "sum"))
        .rename(columns={"faf_orig_str": "src_faf", "faf_dest_str": "dst_faf"})
    )
    return grouped


def demand_edge_index_and_attr(edge_df: pd.DataFrame, node_map: Dict[str, int], reverse: bool = False) -> Tuple[torch.Tensor, torch.Tensor]:
    """Convert demand edge table to edge_index and simple numeric edge attributes."""
    pairs: List[Tuple[str, str]] = []
    attrs: List[List[float]] = []
    for row in edge_df.itertuples(index=False):
        src = normalize_faf_code(getattr(row, "src_faf"))
        dst = normalize_faf_code(getattr(row, "dst_faf"))
        if reverse:
            src, dst = dst, src
        if src not in node_map or dst not in node_map:
            continue
        edge_count = float(getattr(row, "edge_count", 0.0))
        tons_sum = float(getattr(row, "tons_sum", 0.0))
        tmiles_sum = float(getattr(row, "tmiles_sum", 0.0))
        value_sum = float(getattr(row, "value_sum", 0.0))
        pairs.append((src, dst))
        attrs.append([math.log1p(edge_count), math.log1p(max(tons_sum, 0.0)), math.log1p(max(tmiles_sum, 0.0)), math.log1p(max(value_sum, 0.0))])
    edge_index = directed_edge_index_from_pairs(pairs, node_map)
    edge_attr = torch.tensor(attrs, dtype=torch.float32) if attrs else torch.zeros((0, 4), dtype=torch.float32)
    return edge_index, edge_attr


def train_od_edge_table(frame: pd.DataFrame, train_mask: pd.Series) -> pd.DataFrame:
    """Build train-only supervised OD message edge table."""
    subset = frame.loc[train_mask].copy()
    if subset.empty:
        return pd.DataFrame(columns=["src_faf", "dst_faf", "edge_count", "year_count"])
    grouped = (
        subset.groupby(["faf_orig_str", "faf_dest_str"], as_index=False)
        .agg(edge_count=("year_int", "size"), year_count=("year_int", "nunique"))
        .rename(columns={"faf_orig_str": "src_faf", "faf_dest_str": "dst_faf"})
    )
    return grouped


def train_od_edge_index_and_attr(edge_df: pd.DataFrame, node_map: Dict[str, int], reverse: bool = False) -> Tuple[torch.Tensor, torch.Tensor]:
    """Convert train-only OD message edges to edge_index and attributes."""
    pairs: List[Tuple[str, str]] = []
    attrs: List[List[float]] = []
    for row in edge_df.itertuples(index=False):
        src = normalize_faf_code(getattr(row, "src_faf"))
        dst = normalize_faf_code(getattr(row, "dst_faf"))
        if reverse:
            src, dst = dst, src
        if src not in node_map or dst not in node_map:
            continue
        edge_count = float(getattr(row, "edge_count", 0.0))
        year_count = float(getattr(row, "year_count", 0.0))
        pairs.append((src, dst))
        attrs.append([math.log1p(edge_count), year_count])
    edge_index = directed_edge_index_from_pairs(pairs, node_map)
    edge_attr = torch.tensor(attrs, dtype=torch.float32) if attrs else torch.zeros((0, 2), dtype=torch.float32)
    return edge_index, edge_attr


if cfg.message_edge_policy == "cold_train_if_available_else_temporal_train" and supervised_df["cold_split"].eq("train").any():
    message_train_mask = supervised_df["cold_split"].eq("train")
else:
    message_train_mask = supervised_df["temporal_split"].eq("train")

truck_demand_edges = demand_edge_table(supervised_df, "truck", message_train_mask)
rail_demand_edges = demand_edge_table(supervised_df, "rail", message_train_mask)
multimodal_demand_edges = demand_edge_table(supervised_df, "multimodal", message_train_mask)
train_od_edges = train_od_edge_table(supervised_df, message_train_mask)

print("Message train rows:", int(message_train_mask.sum()))
print("Demand edges:", {"truck": len(truck_demand_edges), "rail": len(rail_demand_edges), "multimodal": len(multimodal_demand_edges)})
print("Train OD message edges:", train_od_edges.shape)

Message train rows: 42849
Demand edges: {'truck': 8693, 'rail': 5834, 'multimodal': 8692}
Train OD message edges: (8748, 4)


## 15. Build supervised edge labels and edge-decoder features

The supervised edge-label table contains all OD-year rows. The edge decoder features include:

1. The 64 manifest features.
2. The newly built topology features.

Disruption features are intentionally not included yet. They will be built in a later notebook and appended to this schema.

In [19]:
edge_feature_df = supervised_df[["faf_orig_str", "faf_dest_str", "year_int", "temporal_split", "cold_split"] + LABEL_COLUMNS + manifest_feature_columns].copy()
edge_feature_df.insert(0, "row_id", np.arange(len(edge_feature_df), dtype=np.int64))
edge_feature_df = edge_feature_df.merge(
    topology_features[["row_id"] + topology_feature_columns],
    on="row_id",
    how="left",
)

edge_feature_columns = manifest_feature_columns + topology_feature_columns
for column in edge_feature_columns + LABEL_COLUMNS:
    edge_feature_df[column] = pd.to_numeric(edge_feature_df[column], errors="coerce").fillna(0.0)

# Standardize edge-label attributes for storage. Training notebooks may choose to re-scale them.
edge_attr_values = edge_feature_df[edge_feature_columns].astype(float).to_numpy(dtype=np.float32)
edge_attr_mean = edge_attr_values[edge_feature_df["temporal_split"].eq("train").to_numpy()].mean(axis=0)
edge_attr_std = edge_attr_values[edge_feature_df["temporal_split"].eq("train").to_numpy()].std(axis=0)
edge_attr_std[edge_attr_std < 1e-6] = 1.0
edge_attr_scaled = (edge_attr_values - edge_attr_mean) / edge_attr_std

y_values = edge_feature_df[LABEL_COLUMNS].astype(float).to_numpy(dtype=np.float32)
year_values = edge_feature_df["year_int"].astype(int).to_numpy(dtype=np.int64)

edge_label_pairs = list(zip(edge_feature_df["faf_orig_str"], edge_feature_df["faf_dest_str"]))
edge_label_index = directed_edge_index_from_pairs(edge_label_pairs, zone_to_idx)
if edge_label_index.shape[1] != len(edge_feature_df):
    raise ValueError("Some supervised edge-label rows could not be mapped to FAF node indices.")

print("Supervised edge_label_index shape:", tuple(edge_label_index.shape))
print("Edge decoder feature count:", len(edge_feature_columns))
print("Topology feature count:", len(topology_feature_columns))

Supervised edge_label_index shape: (2, 73972)
Edge decoder feature count: 80
Topology feature count: 16


## 16. Assemble the PyTorch Geometric HeteroData object

This cell creates the full heterogeneous graph. Message-passing relations are stored in `edge_index_dict`, while supervised OD labels and edge-decoder features are stored under the `('faf_zone', 'supervised_od', 'faf_zone')` relation.

In [20]:
def add_relation(data: HeteroData, src_type: str, relation: str, dst_type: str, edge_index: torch.Tensor, edge_attr: Optional[torch.Tensor] = None) -> None:
    """Add a typed relation to a HeteroData object."""
    data[(src_type, relation, dst_type)].edge_index = edge_index.contiguous()
    if edge_attr is not None:
        data[(src_type, relation, dst_type)].edge_attr = edge_attr.contiguous()


hetero_data = HeteroData()
hetero_data["faf_zone"].x = torch.tensor(node_x, dtype=torch.float32)
hetero_data["faf_zone"].node_id = torch.arange(len(all_faf_zones), dtype=torch.long)

hetero_data["terminal"].x = torch.tensor(terminal_x, dtype=torch.float32)
hetero_data["terminal"].node_id = torch.arange(len(terminal_ids), dtype=torch.long)

# Topology edges.
truck_edge_index, truck_edge_attr = topology_edge_index(truck_adj_edges, zone_to_idx, make_bidirectional=True)
rail_edge_index, rail_edge_attr = topology_edge_index(rail_adj_edges, zone_to_idx, make_bidirectional=True)
add_relation(hetero_data, "faf_zone", "truck_adj", "faf_zone", truck_edge_index, truck_edge_attr)
add_relation(hetero_data, "faf_zone", "rail_adj", "faf_zone", rail_edge_index, rail_edge_attr)

# Demand edges.
for mode, table in [("truck", truck_demand_edges), ("rail", rail_demand_edges), ("multimodal", multimodal_demand_edges)]:
    edge_index, edge_attr = demand_edge_index_and_attr(table, zone_to_idx, reverse=False)
    add_relation(hetero_data, "faf_zone", f"demand_{mode}", "faf_zone", edge_index, edge_attr)
    if cfg.include_reverse_demand_edges:
        rev_edge_index, rev_edge_attr = demand_edge_index_and_attr(table, zone_to_idx, reverse=True)
        add_relation(hetero_data, "faf_zone", f"rev_demand_{mode}", "faf_zone", rev_edge_index, rev_edge_attr)

# Train-only OD message edges.
train_edge_index, train_edge_attr = train_od_edge_index_and_attr(train_od_edges, zone_to_idx, reverse=False)
add_relation(hetero_data, "faf_zone", "train_od", "faf_zone", train_edge_index, train_edge_attr)
if cfg.include_reverse_train_od_edges:
    rev_train_edge_index, rev_train_edge_attr = train_od_edge_index_and_attr(train_od_edges, zone_to_idx, reverse=True)
    add_relation(hetero_data, "faf_zone", "rev_train_od", "faf_zone", rev_train_edge_index, rev_train_edge_attr)

# Self loops for FAF zones.
if cfg.include_self_loops:
    self_loop_index = torch.arange(len(all_faf_zones), dtype=torch.long).repeat(2, 1)
    self_loop_attr = torch.ones((len(all_faf_zones), 1), dtype=torch.float32)
    add_relation(hetero_data, "faf_zone", "self_loop", "faf_zone", self_loop_index, self_loop_attr)

# Terminal access edges.
terminal_pairs = []
terminal_attrs = []
for row in terminal_access_edges.itertuples(index=False):
    zone = normalize_faf_code(getattr(row, "src_faf"))
    terminal_id = str(getattr(row, "terminal_id"))
    if zone in zone_to_idx and terminal_id in terminal_to_idx:
        terminal_pairs.append((zone, terminal_id))
        distance = float(getattr(row, "access_distance_miles", 0.0))
        inside = float(bool(getattr(row, "inside_region", False)))
        terminal_attrs.append([distance, math.log1p(max(distance, 0.0)), inside])

if terminal_pairs:
    src_idx = [zone_to_idx[src] for src, _ in terminal_pairs]
    dst_idx = [terminal_to_idx[dst] for _, dst in terminal_pairs]
    access_edge_index = torch.tensor([src_idx, dst_idx], dtype=torch.long)
    rev_access_edge_index = torch.tensor([dst_idx, src_idx], dtype=torch.long)
    access_edge_attr = torch.tensor(terminal_attrs, dtype=torch.float32)
else:
    access_edge_index = torch.empty((2, 0), dtype=torch.long)
    rev_access_edge_index = torch.empty((2, 0), dtype=torch.long)
    access_edge_attr = torch.zeros((0, 3), dtype=torch.float32)

add_relation(hetero_data, "faf_zone", "terminal_access", "terminal", access_edge_index, access_edge_attr)
add_relation(hetero_data, "terminal", "reverse_terminal_access", "faf_zone", rev_access_edge_index, access_edge_attr)

# Supervised edge-label store. This relation is for labels/edge decoder inputs, not message-passing leakage.
supervised_store = hetero_data[("faf_zone", "supervised_od", "faf_zone")]
supervised_store.edge_label_index = edge_label_index.contiguous()
supervised_store.edge_label_attr = torch.tensor(edge_attr_scaled, dtype=torch.float32).contiguous()
supervised_store.edge_label_attr_raw = torch.tensor(edge_attr_values, dtype=torch.float32).contiguous()
supervised_store.y = torch.tensor(y_values, dtype=torch.float32).contiguous()
supervised_store.year = torch.tensor(year_values, dtype=torch.long)
supervised_store.row_id = torch.tensor(edge_feature_df["row_id"].to_numpy(dtype=np.int64), dtype=torch.long)

for split_name in ["train", "val", "test", "unused"]:
    supervised_store[f"temporal_{split_name}_mask"] = torch.tensor(edge_feature_df["temporal_split"].eq(split_name).to_numpy(), dtype=torch.bool)
    supervised_store[f"cold_{split_name}_mask"] = torch.tensor(edge_feature_df["cold_split"].eq(split_name).to_numpy(), dtype=torch.bool)

print(hetero_data)
print("Metadata:", hetero_data.metadata())

HeteroData(
  faf_zone={
    x=[132, 86],
    node_id=[132],
  },
  terminal={
    x=[241, 5],
    node_id=[241],
  },
  (faf_zone, truck_adj, faf_zone)={
    edge_index=[2, 582],
    edge_attr=[582, 3],
  },
  (faf_zone, rail_adj, faf_zone)={
    edge_index=[2, 510],
    edge_attr=[510, 3],
  },
  (faf_zone, demand_truck, faf_zone)={
    edge_index=[2, 8693],
    edge_attr=[8693, 4],
  },
  (faf_zone, rev_demand_truck, faf_zone)={
    edge_index=[2, 8693],
    edge_attr=[8693, 4],
  },
  (faf_zone, demand_rail, faf_zone)={
    edge_index=[2, 5834],
    edge_attr=[5834, 4],
  },
  (faf_zone, rev_demand_rail, faf_zone)={
    edge_index=[2, 5834],
    edge_attr=[5834, 4],
  },
  (faf_zone, demand_multimodal, faf_zone)={
    edge_index=[2, 8692],
    edge_attr=[8692, 4],
  },
  (faf_zone, rev_demand_multimodal, faf_zone)={
    edge_index=[2, 8692],
    edge_attr=[8692, 4],
  },
  (faf_zone, train_od, faf_zone)={
    edge_index=[2, 8748],
    edge_attr=[8748, 2],
  },
  (faf_zone, rev_trai

## 17. Build graph metadata and quality checks

This cell summarizes edge counts, feature dimensions, split masks, and topology coverage. These checks make sure the graph object is usable by downstream D-CQHGT notebooks.

In [21]:
def relation_edge_count(data: HeteroData) -> pd.DataFrame:
    """Return edge counts and edge-attribute dimensions for all relations."""
    rows = []
    for edge_type in data.edge_types:
        store = data[edge_type]
        edge_count = int(store.edge_index.shape[1]) if "edge_index" in store else 0
        edge_attr_dim = int(store.edge_attr.shape[1]) if "edge_attr" in store else 0
        rows.append(
            {
                "src_type": edge_type[0],
                "relation": edge_type[1],
                "dst_type": edge_type[2],
                "edge_count": edge_count,
                "edge_attr_dim": edge_attr_dim,
            }
        )
    return pd.DataFrame(rows)


edge_counts = relation_edge_count(hetero_data)

split_counts = {}
for split_name in ["train", "val", "test", "unused"]:
    split_counts[f"temporal_{split_name}"] = int(supervised_store[f"temporal_{split_name}_mask"].sum())
    split_counts[f"cold_{split_name}"] = int(supervised_store[f"cold_{split_name}_mask"].sum())

topology_coverage = {
    "truck_path_available_rate": float(topology_features["topo_truck_path_available"].mean()),
    "rail_path_available_rate": float(topology_features["topo_rail_path_available"].mean()),
    "both_zones_have_terminal_rate": float(topology_features["topo_both_zones_have_terminal"].mean()),
    "terminal_rail_path_available_rate": float(topology_features["topo_terminal_rail_path_available"].mean()),
}

print("Edge counts:")
print(edge_counts.to_string(index=False))
print("\nSplit counts:")
print(json.dumps(split_counts, indent=2))
print("\nTopology coverage:")
print(json.dumps(topology_coverage, indent=2))

# Sanity checks.
assert hetero_data["faf_zone"].x.shape[0] == len(all_faf_zones)
assert supervised_store.edge_label_index.shape[1] == len(edge_feature_df)
assert supervised_store.y.shape[0] == len(edge_feature_df)
assert split_counts["cold_test"] >= 0

Edge counts:
src_type                relation dst_type  edge_count  edge_attr_dim
faf_zone               truck_adj faf_zone         582              3
faf_zone                rail_adj faf_zone         510              3
faf_zone            demand_truck faf_zone        8693              4
faf_zone        rev_demand_truck faf_zone        8693              4
faf_zone             demand_rail faf_zone        5834              4
faf_zone         rev_demand_rail faf_zone        5834              4
faf_zone       demand_multimodal faf_zone        8692              4
faf_zone   rev_demand_multimodal faf_zone        8692              4
faf_zone                train_od faf_zone        8748              2
faf_zone            rev_train_od faf_zone        8748              2
faf_zone               self_loop faf_zone         132              1
faf_zone         terminal_access terminal         241              3
terminal reverse_terminal_access faf_zone         241              3
faf_zone           su

## 18. Save topology features, node maps, HeteroData, and metadata

This cell writes all output artifacts. The most important files are:

```text
Data/08_processed/graph_inputs/freight_mnet_full_heterodata_east_plus_gulf.pt
Data/08_processed/graph_inputs/topology_features_od_east_plus_gulf.parquet
Data/08_processed/graph_inputs/freight_mnet_full_heterodata_east_plus_gulf.metadata.json
```

Future notebooks should load these files instead of rebuilding topology from scratch.

In [22]:
# Save topology feature table.
topology_export_columns = ["row_id", "faf_orig_str", "faf_dest_str", "year_int", "temporal_split", "cold_split"] + topology_feature_columns
safe_to_parquet(topology_features[topology_export_columns], paths.topology_features_path)

# Save debug tables.
faf_zone_index_df = pd.DataFrame({"faf_zone_str": all_faf_zones, "node_idx": [zone_to_idx[z] for z in all_faf_zones]})
terminal_index_df = pd.DataFrame({"terminal_id": terminal_ids, "node_idx": [terminal_to_idx[t] for t in terminal_ids]})

if cfg.save_debug_tables:
    safe_to_parquet(faf_zone_index_df, paths.tables_dir / "faf_zone_index.parquet")
    safe_to_parquet(terminal_index_df, paths.tables_dir / "terminal_index.parquet")
    safe_to_parquet(faf_node_feature_df, paths.tables_dir / "faf_node_features.parquet")
    safe_to_parquet(terminal_node_df, paths.tables_dir / "terminal_node_features.parquet")
    safe_to_parquet(terminal_access_edges, paths.tables_dir / "terminal_access_edges.parquet")
    safe_to_parquet(truck_adj_edges, paths.tables_dir / "truck_adj_edges.parquet")
    safe_to_parquet(rail_adj_edges, paths.tables_dir / "rail_adj_edges.parquet")
    safe_to_parquet(truck_demand_edges, paths.tables_dir / "demand_truck_edges_train_only.parquet")
    safe_to_parquet(rail_demand_edges, paths.tables_dir / "demand_rail_edges_train_only.parquet")
    safe_to_parquet(multimodal_demand_edges, paths.tables_dir / "demand_multimodal_edges_train_only.parquet")
    safe_to_parquet(train_od_edges, paths.tables_dir / "train_od_message_edges.parquet")
    edge_counts.to_csv(paths.tables_dir / "heterodata_edge_counts.csv", index=False)

# Save HeteroData object.
if cfg.save_heterodata_pt:
    torch.save(hetero_data, paths.heterodata_path)

metadata = {
    "notebook": "Build_Topology_Features_and_Full_HeteroData",
    "config": asdict(cfg),
    "paths": {key: str(value) for key, value in asdict(paths).items()},
    "schema": {
        "node_types": list(hetero_data.node_types),
        "edge_types": [list(edge_type) for edge_type in hetero_data.edge_types],
        "label_columns": LABEL_COLUMNS,
        "manifest_feature_columns": manifest_feature_columns,
        "topology_feature_columns": topology_feature_columns,
        "edge_feature_columns": edge_feature_columns,
        "faf_node_feature_columns": node_feature_columns,
        "terminal_feature_columns": terminal_feature_columns,
    },
    "dimensions": {
        "n_faf_nodes": int(len(all_faf_zones)),
        "n_terminal_nodes": int(len(terminal_ids)),
        "faf_node_feature_dim": int(hetero_data["faf_zone"].x.shape[1]),
        "terminal_node_feature_dim": int(hetero_data["terminal"].x.shape[1]),
        "supervised_edges": int(supervised_store.edge_label_index.shape[1]),
        "edge_label_attr_dim": int(supervised_store.edge_label_attr.shape[1]),
    },
    "split_counts": split_counts,
    "topology_coverage": topology_coverage,
    "edge_counts": edge_counts.to_dict(orient="records"),
    "preprocessing": {
        "edge_attr_mean": edge_attr_mean.tolist(),
        "edge_attr_std": edge_attr_std.tolist(),
        "faf_node_mean": node_mean.tolist(),
        "faf_node_std": node_std.tolist(),
        "terminal_node_mean": terminal_mean.tolist(),
        "terminal_node_std": terminal_std.tolist(),
        "truck_distance_fill_miles": float(truck_fill),
        "rail_distance_fill_miles": float(rail_fill),
    },
    "package_versions": {
        "python": os.sys.version,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "geopandas": gpd.__version__,
        "networkx": nx.__version__,
        "torch": torch.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
    },
}
write_json(metadata, paths.metadata_path)
write_json(metadata, paths.output_dir / "run_metadata_build_topology_full_heterodata.json")

artifact_manifest = {
    "heterodata_pt": str(paths.heterodata_path),
    "topology_features_od": str(paths.topology_features_path),
    "metadata_json": str(paths.metadata_path),
    "tables_dir": str(paths.tables_dir),
    "cache_dir": str(paths.cache_dir),
}
write_json(artifact_manifest, paths.output_dir / "analysis_artifact_manifest_build_topology_full_heterodata.json")

print("Saved topology and HeteroData artifacts:")
print(json.dumps(artifact_manifest, indent=2))

Saved topology and HeteroData artifacts:
{
  "heterodata_pt": "E:\\NetworkOptimization\\pythonProject1\\Data\\08_processed\\graph_inputs\\freight_mnet_full_heterodata_east_plus_gulf.pt",
  "topology_features_od": "E:\\NetworkOptimization\\pythonProject1\\Data\\08_processed\\graph_inputs\\topology_features_od_east_plus_gulf.parquet",
  "metadata_json": "E:\\NetworkOptimization\\pythonProject1\\Data\\08_processed\\graph_inputs\\freight_mnet_full_heterodata_east_plus_gulf.metadata.json",
  "tables_dir": "E:\\NetworkOptimization\\pythonProject1\\Data\\10_experiments\\build_topology_features_full_heterodata_v1\\east_plus_gulf\\tables",
  "cache_dir": "E:\\NetworkOptimization\\pythonProject1\\Data\\10_experiments\\build_topology_features_full_heterodata_v1\\east_plus_gulf\\cache"
}


## 19. Loading test for downstream notebooks

This final cell verifies that the saved PyG object and metadata can be loaded back successfully. Downstream training notebooks should start from this same load pattern.

In [25]:
# ---------------------------------------------------------------------
# Loading test for saved full HeteroData and topology artifacts
# ---------------------------------------------------------------------
#
# This cell validates three saved artifacts:
#   1. PyG HeteroData object
#   2. OD-level topology feature parquet
#   3. metadata JSON
#
# Important implementation detail:
# PyTorch 2.6+ defaults torch.load(..., weights_only=True), but PyG HeteroData
# is a full Python object rather than a plain tensor checkpoint. Therefore,
# locally generated trusted HeteroData files should be loaded with
# weights_only=False.
#
# Another important detail:
# In this notebook, ("faf_zone", "supervised_od", "faf_zone") is a supervised
# label store. It contains edge_label_index and y, not edge_index. Therefore,
# this loading test must handle message-passing edges and supervised label
# edges separately.

import json
from pathlib import Path

import pandas as pd
import torch

try:
    from torch_geometric.data import HeteroData
except Exception as exc:
    raise ImportError(
        "torch_geometric is required to load the saved HeteroData object. "
        "Please activate the freight_mnet CUDA/PyG environment."
    ) from exc


def ensure_artifact_exists(path: Path, description: str) -> None:
    """Raise a clear error if an expected output artifact is missing."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{description} was not found: {path}")


def load_saved_heterodata(path: Path) -> HeteroData:
    """
    Load a PyG HeteroData object saved by this notebook.

    Use weights_only=False only for trusted local artifacts generated by this
    notebook. Do not use this setting for untrusted external files.
    """
    path = Path(path)
    ensure_artifact_exists(path, "Saved HeteroData .pt file")

    try:
        data_object = torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )
    except TypeError:
        # Compatibility fallback for older PyTorch versions.
        data_object = torch.load(
            path,
            map_location="cpu",
        )

    if not isinstance(data_object, HeteroData):
        raise TypeError(
            "The loaded object is not a torch_geometric.data.HeteroData. "
            f"Loaded type: {type(data_object)}"
        )

    return data_object


def storage_has_key(storage, key: str) -> bool:
    """Return True if a PyG storage object contains a specific attribute key."""
    try:
        return key in storage.keys()
    except Exception:
        return hasattr(storage, key)


def tensor_shape_or_none(storage, key: str):
    """Return tensor shape for a PyG storage key, or None if the key is absent."""
    if not storage_has_key(storage, key):
        return None

    value = storage[key]
    if hasattr(value, "shape"):
        return tuple(value.shape)

    return None


def print_node_store_summary(data: HeteroData) -> None:
    """Print node types and node-level tensor shapes."""
    print("\nNode store summary:")
    for node_type in data.node_types:
        store = data[node_type]
        keys = list(store.keys())

        print(f"  Node type: {node_type}")
        print(f"    keys: {keys}")

        for key in keys:
            shape = tensor_shape_or_none(store, key)
            if shape is not None:
                print(f"    {key}: shape = {shape}")


def print_edge_store_summary(data: HeteroData) -> None:
    """
    Print message-passing edge stores and supervised label stores separately.

    Message-passing edge stores contain edge_index.
    Supervised label stores contain edge_label_index and y.
    """
    print("\nEdge store summary:")

    message_edge_types = []
    supervised_edge_types = []
    other_edge_types = []

    for edge_type in data.edge_types:
        store = data[edge_type]

        if storage_has_key(store, "edge_index"):
            message_edge_types.append(edge_type)
        elif storage_has_key(store, "edge_label_index"):
            supervised_edge_types.append(edge_type)
        else:
            other_edge_types.append(edge_type)

    print("\nMessage-passing edge types:")
    for edge_type in message_edge_types:
        store = data[edge_type]

        edge_index_shape = tensor_shape_or_none(store, "edge_index")
        edge_attr_shape = tensor_shape_or_none(store, "edge_attr")

        print(f"  {edge_type}")
        print(f"    edge_index: shape = {edge_index_shape}")
        if edge_attr_shape is not None:
            print(f"    edge_attr:   shape = {edge_attr_shape}")

    print("\nSupervised label edge types:")
    for edge_type in supervised_edge_types:
        store = data[edge_type]

        print(f"  {edge_type}")
        print(f"    keys: {list(store.keys())}")

        for key in [
            "edge_label_index",
            "edge_label_attr",
            "edge_label_attr_raw",
            "y",
            "year",
            "row_id",
            "temporal_train_mask",
            "temporal_val_mask",
            "temporal_test_mask",
            "cold_train_mask",
            "cold_val_mask",
            "cold_test_mask",
            "cold_unused_mask",
        ]:
            shape = tensor_shape_or_none(store, key)
            if shape is not None:
                print(f"    {key}: shape = {shape}")

    if other_edge_types:
        print("\nOther edge stores without edge_index or edge_label_index:")
        for edge_type in other_edge_types:
            store = data[edge_type]
            print(f"  {edge_type}: keys = {list(store.keys())}")


def validate_expected_schema(data: HeteroData) -> None:
    """Run lightweight schema checks for the full freight HeteroData object."""
    expected_node_types = {"faf_zone", "terminal"}
    actual_node_types = set(data.node_types)

    missing_node_types = expected_node_types - actual_node_types
    if missing_node_types:
        print("Warning: missing expected node types:", sorted(missing_node_types))
    else:
        print("\nNode type check passed.")

    expected_message_relations = {
        "train_od",
        "rev_train_od",
        "demand_truck",
        "rev_demand_truck",
        "demand_rail",
        "rev_demand_rail",
        "demand_multimodal",
        "rev_demand_multimodal",
        "truck_adj",
        "rail_adj",
        "terminal_access",
        "reverse_terminal_access",
        "self_loop",
    }

    actual_message_relations = {
        edge_type[1]
        for edge_type in data.edge_types
        if storage_has_key(data[edge_type], "edge_index")
    }

    missing_message_relations = expected_message_relations - actual_message_relations
    if missing_message_relations:
        print(
            "Warning: missing expected message-passing edge relations:",
            sorted(missing_message_relations),
        )
    else:
        print("Message-passing edge relation check passed.")

    supervised_edge_type = ("faf_zone", "supervised_od", "faf_zone")
    if supervised_edge_type not in data.edge_types:
        print("Warning: supervised_od edge store is missing.")
    else:
        supervised_store = data[supervised_edge_type]
        required_supervised_keys = {
            "edge_label_index",
            "edge_label_attr",
            "y",
            "temporal_train_mask",
            "temporal_val_mask",
            "temporal_test_mask",
            "cold_train_mask",
            "cold_val_mask",
            "cold_test_mask",
        }

        missing_supervised_keys = {
            key for key in required_supervised_keys
            if not storage_has_key(supervised_store, key)
        }

        if missing_supervised_keys:
            print(
                "Warning: supervised_od store is missing keys:",
                sorted(missing_supervised_keys),
            )
        else:
            print("Supervised edge-label store check passed.")

        edge_label_index_shape = tensor_shape_or_none(supervised_store, "edge_label_index")
        y_shape = tensor_shape_or_none(supervised_store, "y")

        if edge_label_index_shape is not None and y_shape is not None:
            n_label_edges = edge_label_index_shape[1]
            n_y = y_shape[0]
            if n_label_edges != n_y:
                print(
                    "Warning: edge_label_index and y have inconsistent row counts:",
                    edge_label_index_shape,
                    y_shape,
                )
            else:
                print(f"Supervised label count check passed: {n_y:,} labels.")


# ---------------------------------------------------------------------
# Load and validate saved HeteroData.
# ---------------------------------------------------------------------

if cfg.save_heterodata_pt:
    loaded_data = load_saved_heterodata(paths.heterodata_path)

    print("Loaded HeteroData successfully.")
    print(loaded_data)

    print("\nNode types:", loaded_data.node_types)
    print("Edge types:", loaded_data.edge_types)

    print_node_store_summary(loaded_data)
    print_edge_store_summary(loaded_data)
    validate_expected_schema(loaded_data)


# ---------------------------------------------------------------------
# Load and validate topology feature table.
# ---------------------------------------------------------------------

ensure_artifact_exists(paths.topology_features_path, "Saved topology feature parquet")

loaded_topology = pd.read_parquet(paths.topology_features_path)

print("\nLoaded topology features:", loaded_topology.shape)
print("Topology feature columns:")
for column in loaded_topology.columns:
    print(f"  - {column}")

expected_topology_columns = {
    "topo_truck_distance_miles",
    "topo_rail_distance_miles",
    "topo_truck_path_available",
    "topo_rail_path_available",
    "topo_truck_rail_distance_ratio",
    "topo_origin_terminal_count",
    "topo_dest_terminal_count",
    "topo_origin_nearest_terminal_distance_miles",
    "topo_dest_nearest_terminal_distance_miles",
    "topo_both_zones_have_terminal",
    "topo_same_faf_zone",
}

missing_topology_columns = expected_topology_columns - set(loaded_topology.columns)
if missing_topology_columns:
    print(
        "\nWarning: missing expected topology feature columns:",
        sorted(missing_topology_columns),
    )
else:
    print("\nTopology feature column check passed.")

print("\nTopology feature preview:")
print(loaded_topology.head().to_string(index=False))


# ---------------------------------------------------------------------
# Load and validate metadata JSON.
# ---------------------------------------------------------------------

ensure_artifact_exists(paths.metadata_path, "Saved metadata JSON")

with paths.metadata_path.open("r", encoding="utf-8") as file:
    loaded_metadata = json.load(file)

print("\nLoaded metadata keys:", sorted(loaded_metadata.keys()))

if "edge_counts" in loaded_metadata:
    print("\nMetadata edge counts:")
    for key, value in loaded_metadata["edge_counts"].items():
        print(f"  {key}: {value}")

if "node_counts" in loaded_metadata:
    print("\nMetadata node counts:")
    for key, value in loaded_metadata["node_counts"].items():
        print(f"  {key}: {value}")

print("\nLoading test completed successfully.")

Loaded HeteroData successfully.
HeteroData(
  faf_zone={
    x=[132, 86],
    node_id=[132],
  },
  terminal={
    x=[241, 5],
    node_id=[241],
  },
  (faf_zone, truck_adj, faf_zone)={
    edge_index=[2, 582],
    edge_attr=[582, 3],
  },
  (faf_zone, rail_adj, faf_zone)={
    edge_index=[2, 510],
    edge_attr=[510, 3],
  },
  (faf_zone, demand_truck, faf_zone)={
    edge_index=[2, 8693],
    edge_attr=[8693, 4],
  },
  (faf_zone, rev_demand_truck, faf_zone)={
    edge_index=[2, 8693],
    edge_attr=[8693, 4],
  },
  (faf_zone, demand_rail, faf_zone)={
    edge_index=[2, 5834],
    edge_attr=[5834, 4],
  },
  (faf_zone, rev_demand_rail, faf_zone)={
    edge_index=[2, 5834],
    edge_attr=[5834, 4],
  },
  (faf_zone, demand_multimodal, faf_zone)={
    edge_index=[2, 8692],
    edge_attr=[8692, 4],
  },
  (faf_zone, rev_demand_multimodal, faf_zone)={
    edge_index=[2, 8692],
    edge_attr=[8692, 4],
  },
  (faf_zone, train_od, faf_zone)={
    edge_index=[2, 8748],
    edge_attr=[8748

AttributeError: 'list' object has no attribute 'items'

In [26]:
# ---------------------------------------------------------------------
# Robust metadata printing
# ---------------------------------------------------------------------
#
# Some metadata fields are dictionaries, while others are lists of records.
# This helper prints both safely.

def print_metadata_field(metadata: dict, field_name: str, max_items: int = 20) -> None:
    """Print a metadata field that may be a dict, list, or scalar."""
    if field_name not in metadata:
        print(f"\nMetadata field '{field_name}' is not present.")
        return

    value = metadata[field_name]
    print(f"\nMetadata {field_name}:")

    if isinstance(value, dict):
        for key, item in value.items():
            print(f"  {key}: {item}")
        return

    if isinstance(value, list):
        print(f"  list length: {len(value)}")
        for idx, item in enumerate(value[:max_items]):
            print(f"  [{idx}] {item}")
        if len(value) > max_items:
            print(f"  ... {len(value) - max_items} more items")
        return

    print(f"  {value}")


print("\nLoaded metadata keys:", sorted(loaded_metadata.keys()))

for field_name in [
    "node_counts",
    "edge_counts",
    "dimensions",
    "split_counts",
    "topology_coverage",
    "schema",
    "preprocessing",
]:
    print_metadata_field(loaded_metadata, field_name)

print("\nLoading test completed successfully.")


Loaded metadata keys: ['config', 'dimensions', 'edge_counts', 'notebook', 'package_versions', 'paths', 'preprocessing', 'schema', 'split_counts', 'topology_coverage']

Metadata field 'node_counts' is not present.

Metadata edge_counts:
  list length: 14
  [0] {'src_type': 'faf_zone', 'relation': 'truck_adj', 'dst_type': 'faf_zone', 'edge_count': 582, 'edge_attr_dim': 3}
  [1] {'src_type': 'faf_zone', 'relation': 'rail_adj', 'dst_type': 'faf_zone', 'edge_count': 510, 'edge_attr_dim': 3}
  [2] {'src_type': 'faf_zone', 'relation': 'demand_truck', 'dst_type': 'faf_zone', 'edge_count': 8693, 'edge_attr_dim': 4}
  [3] {'src_type': 'faf_zone', 'relation': 'rev_demand_truck', 'dst_type': 'faf_zone', 'edge_count': 8693, 'edge_attr_dim': 4}
  [4] {'src_type': 'faf_zone', 'relation': 'demand_rail', 'dst_type': 'faf_zone', 'edge_count': 5834, 'edge_attr_dim': 4}
  [5] {'src_type': 'faf_zone', 'relation': 'rev_demand_rail', 'dst_type': 'faf_zone', 'edge_count': 5834, 'edge_attr_dim': 4}
  [6] {'sr

## 20. Interpretation checklist

After this notebook runs successfully, check the following before training `DCQHGT_Topology_ColdOD`:

1. `heterodata_edge_counts.csv` has non-zero truck, rail, demand, and terminal access relations.
2. `topology_coverage` in the metadata is not near zero for both truck and rail paths.
3. Cold-OD split masks match the previous Cold-OD baseline experiment.
4. The supervised edge-label relation has exactly the same number of rows as the model-ready supervised table.
5. The topology feature parquet contains no object columns except identifiers and split labels.

The next modeling notebook should compare:

- FAF-zone HGT baseline.
- HGT with terminal nodes only.
- HGT with truck/rail topology only.
- HGT with terminal access only.
- HGT with topology edge-decoder features.
- Full topology D-CQHGT without disruption gate.